#### Importing Libraries

In [1]:
import json
import joblib
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn

from pathlib import Path

from sklearn.base import clone
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import ParameterGrid

#### Loading Feature-Engineered Datasets

In [2]:
data_dir = Path("../data/processed")
artifacts_dir = Path("../artifacts")
models_dir = Path("../models")

models_dir.mkdir(parents=True, exist_ok=True)

train_df = pd.read_parquet(
    data_dir / "forecast_features_train.parquet"
)

test_df = pd.read_parquet(
    data_dir / "forecast_features_test.parquet"
)

with open(
    artifacts_dir / "forecast_feature_config.json",
    "r",
    encoding="utf-8"
) as file:
    feature_config = json.load(file)

forecast_horizon = feature_config["forecast_horizon"]

horizon_feature_columns = {
    int(horizon): columns
    for horizon, columns in feature_config["horizon_features"].items()
}

horizon_target_columns = {
    int(horizon): target
    for horizon, target in feature_config["horizon_targets"].items()
}

training_end_date = pd.Timestamp(
    feature_config["training_end_date"]
)

assert train_df["Target_Date_H7"].le(training_end_date).all()
assert test_df["Target_Date_H1"].gt(training_end_date).all()

assert set(horizon_feature_columns) == set(range(1, forecast_horizon + 1))
assert set(horizon_target_columns) == set(range(1, forecast_horizon + 1))

print("Training Dataset Shape:", train_df.shape)
print("Testing Dataset Shape:", test_df.shape)
print("Forecast Horizon:", forecast_horizon)
print("Training Cutoff:", training_end_date.date())

print("\nForecasting Dataset Summary:")
display(
    pd.DataFrame({
        "Dataset": ["Training", "Testing"],
        "Observations": [len(train_df), len(test_df)],
        "First_Origin": [
            train_df["Forecast_Origin"].min(),
            test_df["Forecast_Origin"].min()
        ],
        "Last_Origin": [
            train_df["Forecast_Origin"].max(),
            test_df["Forecast_Origin"].max()
        ]
    })
)

print("\nHorizon-Specific Predictor Counts:")
display(
    pd.DataFrame({
        "Horizon": list(horizon_feature_columns),
        "Predictors": [
            len(columns)
            for columns in horizon_feature_columns.values()
        ],
        "Target": [
            horizon_target_columns[horizon]
            for horizon in horizon_feature_columns
        ]
    })
)

Training Dataset Shape: (207, 87)
Testing Dataset Shape: (55, 87)
Forecast Horizon: 7
Training Cutoff: 2011-09-29

Forecasting Dataset Summary:


,Dataset,Observations,First_Origin,Last_Origin
0,Training,207,2011-01-16,2011-09-21
1,Testing,55,2011-09-29,2011-12-01



Horizon-Specific Predictor Counts:


,Horizon,Predictors,Target
0,1,48,Target_H1
1,2,48,Target_H2
2,3,48,Target_H3
3,4,48,Target_H4
4,5,48,Target_H5
5,6,48,Target_H6
6,7,48,Target_H7


**Key Observations**
- **Modeling Datasets:** The training dataset contains 207 forecasting observations, while the testing dataset contains 55, with 87 columns each and a chronological training cutoff of September 29, 2011.
- **Forecasting Configuration:** All seven forecast horizons have been loaded successfully, each containing 48 predictors and a corresponding revenue target.

#### Configuring Expanding-Window Validation

In [3]:
n_splits = 3
validation_size = 14

validation_folds = []

n_observations = len(train_df)
first_validation_start = n_observations - n_splits * validation_size

for fold in range(n_splits):
    validation_start = first_validation_start + fold * validation_size
    validation_end = validation_start + validation_size

    validation_indices = np.arange(
        validation_start,
        validation_end
    )

    first_validation_origin = train_df.loc[
        validation_start, "Forecast_Origin"
    ]

    eligible_training_indices = np.flatnonzero(
        train_df["Target_Date_H7"].le(first_validation_origin).to_numpy()
    )

    assert len(eligible_training_indices) > 0
    assert eligible_training_indices.max() < validation_start

    assert (
        train_df.loc[
            eligible_training_indices, "Target_Date_H7"
        ].max() <= first_validation_origin
    )

    validation_folds.append(
        (eligible_training_indices, validation_indices)
    )

fold_summary = pd.DataFrame([
    {
        "Fold": fold_number,
        "Training_Observations": len(training_indices),
        "Validation_Observations": len(validation_indices),
        "Training_Last_Origin": train_df.loc[
            training_indices[-1], "Forecast_Origin"
        ],
        "Training_Last_Target": train_df.loc[
            training_indices[-1], "Target_Date_H7"
        ],
        "Validation_First_Origin": train_df.loc[
            validation_indices[0], "Forecast_Origin"
        ],
        "Validation_Last_Origin": train_df.loc[
            validation_indices[-1], "Forecast_Origin"
        ]
    }
    for fold_number, (training_indices, validation_indices)
    in enumerate(validation_folds, start=1)
])

print("Expanding-Window Validation Configuration:")
print("Number of Folds:", len(validation_folds))
print("Validation Observations per Fold:", validation_size)

print("\nFold Summary:")
display(fold_summary)

print("\nValidation Checks:")
assert all(
    train_df.loc[training_indices, "Target_Date_H7"].max()
    <= train_df.loc[validation_indices[0], "Forecast_Origin"]
    for training_indices, validation_indices in validation_folds
)

print("All validation folds passed chronological leakage checks.")

Expanding-Window Validation Configuration:
Number of Folds: 3
Validation Observations per Fold: 14

Fold Summary:


,Fold,Training_Observations,Validation_Observations,Training_Last_Origin,Training_Last_Target,Validation_First_Origin,Validation_Last_Origin
0,1,159,14,2011-07-26,2011-08-03,2011-08-03,2011-08-18
1,2,173,14,2011-08-11,2011-08-19,2011-08-19,2011-09-05
2,3,187,14,2011-08-28,2011-09-06,2011-09-06,2011-09-21



Validation Checks:
All validation folds passed chronological leakage checks.


**Key Observations**
- **Expanding-Window Validation:** Three validation folds were established, each containing 14 forecast origins, with training observations increasing from 159 to 187 across successive folds.
- **Leakage Prevention:** All folds passed chronological validation checks, ensuring that training targets do not extend beyond the first validation forecast origin.

#### Establishing Baseline Forecasting Models

In [5]:
baseline_predictions = []

seasonal_period = 6
revenue_history = pd.concat(
    [
        train_df[["Forecast_Origin", "Origin_Revenue"]],
        test_df[["Forecast_Origin", "Origin_Revenue"]]
    ],
    ignore_index=True
).drop_duplicates(subset="Forecast_Origin")

revenue_history = (
    revenue_history
    .sort_values("Forecast_Origin")
    .reset_index(drop=True)
)

for fold_number, (training_indices, validation_indices) in enumerate(
    validation_folds,
    start=1
):
    validation_data = train_df.iloc[validation_indices]

    for horizon in range(1, forecast_horizon + 1):
        actual_values = validation_data[
            horizon_target_columns[horizon]
        ].to_numpy()

        naive_predictions = validation_data[
            "Origin_Revenue"
        ].to_numpy()

        seasonal_lag = (seasonal_period - horizon % seasonal_period) % seasonal_period

        if seasonal_lag == 0:
            seasonal_predictions = validation_data[
                "Origin_Revenue"
            ].to_numpy()
        else:
            seasonal_predictions = revenue_history[
                "Origin_Revenue"
            ].shift(seasonal_lag).reindex(
                validation_data.index
            ).to_numpy()

        for model_name, predictions in [
            ("Naive", naive_predictions),
            ("Seasonal_Naive", seasonal_predictions)
        ]:
            baseline_predictions.append({
                "Fold": fold_number,
                "Horizon": horizon,
                "Model": model_name,
                "MAE": mean_absolute_error(
                    actual_values, predictions
                ),
                "RMSE": np.sqrt(
                    mean_squared_error(
                        actual_values, predictions
                    )
                )
            })

baseline_results_df = pd.DataFrame(baseline_predictions)

baseline_summary_df = (
    baseline_results_df
    .groupby("Model", as_index=False)
    .agg(
        Mean_MAE=("MAE", "mean"),
        Mean_RMSE=("RMSE", "mean")
    )
    .sort_values("Mean_MAE")
    .reset_index(drop=True)
)

print("Baseline Forecasting Performance:")
display(baseline_summary_df.round(2))

print("\nBaseline Performance by Horizon:")
display(
    baseline_results_df
    .groupby(["Model", "Horizon"], as_index=False)
    .agg(
        Mean_MAE=("MAE", "mean"),
        Mean_RMSE=("RMSE", "mean")
    )
    .round(2)
)

print("\nBaseline Evaluation Completed Successfully.")

Baseline Forecasting Performance:


,Model,Mean_MAE,Mean_RMSE
0,Seasonal_Naive,13194.90,17086.83
1,Naive,18658.22,24089.94



Baseline Performance by Horizon:


,Model,Horizon,Mean_MAE,Mean_RMSE
0,Naive,1,20029.21,25075.50
1,Naive,2,18880.23,25936.44
2,Naive,3,20767.65,26856.53
3,Naive,4,19856.72,25159.75
4,Naive,5,17970.71,23556.93
5,Naive,6,14218.33,18407.34
6,Naive,7,18884.66,23637.07
7,Seasonal_Naive,1,12259.32,15899.58
8,Seasonal_Naive,2,12401.03,16054.24
9,Seasonal_Naive,3,12696.48,16232.43



Baseline Evaluation Completed Successfully.


**Key Observations**
- **Baseline Performance:** Seasonal Naive achieved the lowest average forecasting errors, with an MAE of 13,194.90 and RMSE of 17,086.83, outperforming the Naive baseline (MAE: 18,658.22; RMSE: 24,089.94).
- **Seasonal Predictive Value:** Seasonal Naive reduced average MAE by approximately 29.3% compared with Naive forecasting, indicating that historical six-trading-day revenue patterns provide a useful benchmark for subsequent machine learning models.

#### Visualizing Baseline Forecasting Performance

In [6]:
import plotly.express as px

baseline_comparison_df = (
    baseline_results_df
    .groupby(["Model", "Horizon"], as_index=False)
    .agg(
        Mean_MAE=("MAE", "mean"),
        Mean_RMSE=("RMSE", "mean")
    )
)

fig = px.line(
    baseline_comparison_df,
    x="Horizon",
    y="Mean_MAE",
    color="Model",
    markers=True,
    title="Baseline Forecasting Error Across Prediction Horizons",
    labels={
        "Horizon": "Forecast Horizon (Observed Trading Days)",
        "Mean_MAE": "Mean Absolute Error",
        "Model": "Baseline Model"
    },
    template="plotly_white"
)

fig.update_layout(
    height=500,
    hovermode="x unified",
    legend_title_text="Baseline Model"
)

fig.update_xaxes(
    dtick=1
)

fig.show()

print("Baseline Forecasting Performance by Horizon:")
display(
    baseline_comparison_df
    .sort_values(["Model", "Horizon"])
    .round(2)
)

Baseline Forecasting Performance by Horizon:


,Model,Horizon,Mean_MAE,Mean_RMSE
0,Naive,1,20029.21,25075.50
1,Naive,2,18880.23,25936.44
2,Naive,3,20767.65,26856.53
3,Naive,4,19856.72,25159.75
4,Naive,5,17970.71,23556.93
5,Naive,6,14218.33,18407.34
6,Naive,7,18884.66,23637.07
7,Seasonal_Naive,1,12259.32,15899.58
8,Seasonal_Naive,2,12401.03,16054.24
9,Seasonal_Naive,3,12696.48,16232.43


**Key Observations**
- **Consistent Seasonal Advantage:** Seasonal Naive achieves lower MAE and RMSE than the standard Naive baseline across all seven forecast horizons, demonstrating the predictive value of recurring six-trading-day revenue patterns.
- **Horizon-Wise Performance:** Seasonal Naive records its lowest MAE at Horizon 1 (12,259.32) and highest at Horizon 5 (14,210.94), indicating moderate variation in forecasting accuracy across the prediction horizon.

#### Configuring Machine Learning Models

In [7]:
random_state = 42

models = {
    "Ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=10.0))
    ]),
    "ElasticNet": Pipeline([
        ("scaler", StandardScaler()),
        ("model", ElasticNet(
            alpha=1.0,
            l1_ratio=0.5,
            max_iter=10000,
            random_state=random_state
        ))
    ]),
    "Random_Forest": RandomForestRegressor(
        n_estimators=150,
        max_depth=12,
        min_samples_leaf=3,
        random_state=random_state,
        n_jobs=-1
    ),
    "Extra_Trees": ExtraTreesRegressor(
        n_estimators=150,
        max_depth=12,
        min_samples_leaf=3,
        random_state=random_state,
        n_jobs=-1
    ),
    "Hist_Gradient_Boosting": HistGradientBoostingRegressor(
        max_iter=150,
        learning_rate=0.05,
        max_leaf_nodes=15,
        min_samples_leaf=10,
        random_state=random_state
    )
}

model_configuration = pd.DataFrame([
    {
        "Model": model_name,
        "Estimator": type(model).__name__,
        "Requires_Scaling": isinstance(model, Pipeline)
    }
    for model_name, model in models.items()
])

print("Configured Machine Learning Models:")
display(model_configuration)

print("\nTotal Models:", len(models))
print("Forecast Horizons:", forecast_horizon)
print("Total Model-Horizon Combinations:", len(models) * forecast_horizon)

Configured Machine Learning Models:


,Model,Estimator,Requires_Scaling
0,Ridge,Pipeline,True
1,ElasticNet,Pipeline,True
2,Random_Forest,RandomForestRegressor,False
3,Extra_Trees,ExtraTreesRegressor,False
4,Hist_Gradient_Boosting,HistGradientBoostingRegressor,False



Total Models: 5
Forecast Horizons: 7
Total Model-Horizon Combinations: 35


**Key Observations**
- **Model Diversity:** Five regression algorithms have been configured, covering regularized linear models, tree-based ensembles, and gradient boosting to compare different approaches to revenue forecasting.
- **Horizon-Specific Modeling:** Each algorithm will train seven independent forecasting models, resulting in 35 model-horizon combinations. Feature scaling is applied only to the linear models through preprocessing pipelines.

#### Evaluating Machine Learning Models Using Expanding-Window Validation

In [9]:
model_validation_results = []

for model_name, model in models.items():
    print(f"Evaluating {model_name}...")

    for horizon in range(1, forecast_horizon + 1):
        feature_columns = horizon_feature_columns[horizon]
        target_column = horizon_target_columns[horizon]

        for fold_number, (train_indices, validation_indices) in enumerate(validation_folds, start=1):
            X_train_fold = train_df.iloc[train_indices][feature_columns]
            y_train_fold = train_df.iloc[train_indices][target_column]

            X_validation_fold = train_df.iloc[validation_indices][feature_columns]
            y_validation_fold = train_df.iloc[validation_indices][target_column]

            estimator = clone(model)
            estimator.fit(X_train_fold, y_train_fold)

            predictions = estimator.predict(X_validation_fold)

            model_validation_results.append({
                "Model": model_name,
                "Horizon": horizon,
                "Fold": fold_number,
                "MAE": mean_absolute_error(y_validation_fold, predictions),
                "RMSE": np.sqrt(mean_squared_error(y_validation_fold, predictions)),
                "R2": r2_score(y_validation_fold, predictions)
            })

model_validation_results_df = pd.DataFrame(model_validation_results)

model_validation_summary_df = (
    model_validation_results_df
    .groupby("Model", as_index=False)
    .agg(
        Mean_MAE=("MAE", "mean"),
        Mean_RMSE=("RMSE", "mean"),
        Mean_R2=("R2", "mean")
    )
    .sort_values("Mean_MAE")
    .reset_index(drop=True)
)

print("\nMachine Learning Validation Performance:")
display(model_validation_summary_df.round(4))

print("\nTotal Validation Results:", len(model_validation_results_df))
print("Total Model Fits:", len(model_validation_results_df))

Evaluating Ridge...
Evaluating ElasticNet...
Evaluating Random_Forest...
Evaluating Extra_Trees...
Evaluating Hist_Gradient_Boosting...

Machine Learning Validation Performance:


,Model,Mean_MAE,Mean_RMSE,Mean_R2
0,Random_Forest,12008.9388,16806.5875,-0.0111
1,ElasticNet,12078.3441,16616.4295,0.0322
2,Extra_Trees,12269.2499,16949.4003,-0.0539
3,Hist_Gradient_Boosting,12460.0799,17179.9529,-0.0751
4,Ridge,12669.5148,17215.4998,-0.0663



Total Validation Results: 105
Total Model Fits: 105


**Key Observations**
- **Model Performance:** Random Forest achieved the lowest average MAE (12,008.94), outperforming the Seasonal Naive baseline (13,194.90) by approximately 9.0%. ElasticNet achieved the lowest RMSE (16,616.43) and highest mean R² (0.0322).
- **Predictive Limitations:** Most models produced slightly negative mean R² values, indicating difficulty explaining revenue variation across validation periods despite improvements in absolute forecasting error.

#### Comparing Model Performance Across Forecast Horizons

In [10]:
horizon_performance_df = (
    model_validation_results_df
    .groupby(["Model", "Horizon"], as_index=False)
    .agg(
        Mean_MAE=("MAE", "mean"),
        Mean_RMSE=("RMSE", "mean"),
        Mean_R2=("R2", "mean")
    )
)

fig = px.line(
    horizon_performance_df,
    x="Horizon",
    y="Mean_MAE",
    color="Model",
    markers=True,
    title="Machine Learning Forecasting Error Across Prediction Horizons",
    labels={
        "Horizon": "Forecast Horizon (Observed Trading Days)",
        "Mean_MAE": "Mean Absolute Error",
        "Model": "Machine Learning Model"
    },
    template="plotly_white"
)

fig.update_layout(
    height=550,
    hovermode="x unified"
)

fig.update_xaxes(dtick=1)

fig.show()

print("Machine Learning Performance by Forecast Horizon:")
display(
    horizon_performance_df
    .sort_values(["Horizon", "Mean_MAE"])
    .round(4)
)

Machine Learning Performance by Forecast Horizon:


,Model,Horizon,Mean_MAE,Mean_RMSE,Mean_R2
0,ElasticNet,1,12350.1490,16978.8571,0.0989
21,Random_Forest,1,12490.7783,17186.9949,0.0808
7,Extra_Trees,1,12628.4168,17318.4342,0.0517
28,Ridge,1,12684.5602,17231.8268,0.0486
14,Hist_Gradient_Boosting,1,13411.1803,18281.8638,-0.0524
22,Random_Forest,2,12253.2206,17344.0913,-0.0231
1,ElasticNet,2,12456.6541,17517.6153,-0.0363
8,Extra_Trees,2,12816.6935,17574.3498,-0.0540
29,Ridge,2,13148.0425,18850.3154,-0.2355
15,Hist_Gradient_Boosting,2,13555.8309,18401.8488,-0.1716


**Key Observations**
- **Horizon-Specific Performance:** ElasticNet achieved the lowest MAE at Horizons 1, 4, and 7; Random Forest led at Horizons 2 and 5; and Histogram Gradient Boosting performed best at Horizons 3 and 6.
- **Forecasting Consistency:** Model rankings vary across forecast horizons, with several negative R² values indicating difficulty capturing revenue fluctuations. Random Forest remains the strongest overall model by average MAE, while ElasticNet provides competitive performance and the lowest overall RMSE.

#### Configuring Additional Machine Learning Models

In [13]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

additional_models = {
    "Linear_Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),
    "Lasso": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Lasso(
            alpha=100.0,
            max_iter=20000,
            random_state=42
        ))
    ]),
    "Decision_Tree": DecisionTreeRegressor(
        max_depth=6,
        min_samples_leaf=5,
        random_state=42
    ),
    "Gradient_Boosting": GradientBoostingRegressor(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),
    "XGBoost": XGBRegressor(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        tree_method="hist",
        n_jobs=4,
        random_state=42
    ),
    "LightGBM": LGBMRegressor(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=5,
        num_leaves=15,
        min_child_samples=10,
        verbosity=-1,
        n_jobs=4,
        random_state=42
    ),
    "SVR_RBF": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVR(
            kernel="rbf",
            C=10000,
            epsilon=1000,
            gamma="scale"
        ))
    ])
}

additional_model_configuration = pd.DataFrame([
    {
        "Model": model_name,
        "Estimator": type(model).__name__,
        "Requires_Scaling": isinstance(model, Pipeline)
    }
    for model_name, model in additional_models.items()
])

print("Additional Machine Learning Models:")
display(additional_model_configuration)

print("\nPreviously Evaluated Models:", len(models))
print("Additional Models:", len(additional_models))
print("Total Machine Learning Algorithms:", len(models) + len(additional_models))
print("Additional Model-Horizon Combinations:", len(additional_models) * forecast_horizon)
print("Additional Validation Fits:", len(additional_models) * forecast_horizon * len(validation_folds))

Additional Machine Learning Models:


,Model,Estimator,Requires_Scaling
0,Linear_Regression,Pipeline,True
1,Lasso,Pipeline,True
2,Decision_Tree,DecisionTreeRegressor,False
3,Gradient_Boosting,GradientBoostingRegressor,False
4,XGBoost,XGBRegressor,False
5,LightGBM,LGBMRegressor,False
6,SVR_RBF,Pipeline,True



Previously Evaluated Models: 5
Additional Models: 7
Total Machine Learning Algorithms: 12
Additional Model-Horizon Combinations: 49
Additional Validation Fits: 147


**Key Observations**
- **Expanded Model Coverage:** Seven additional regression algorithms have been configured, bringing the total to 12 classical machine learning algorithms covering linear, tree-based, boosting, and kernel-based approaches.
- **Validation Scope:** The additional algorithms require 147 model fits across seven forecast horizons and three expanding-window validation folds, maintaining a consistent evaluation framework without accessing the final test dataset.

#### Evaluating Additional Machine Learning Models

In [14]:
additional_validation_results = []

for model_name, model in additional_models.items():
    print(f"Evaluating {model_name}...")

    for horizon in range(1, forecast_horizon + 1):
        feature_columns = horizon_feature_columns[horizon]
        target_column = horizon_target_columns[horizon]

        for fold_number, (train_indices, validation_indices) in enumerate(validation_folds, start=1):
            X_train_fold = train_df.iloc[train_indices][feature_columns]
            y_train_fold = train_df.iloc[train_indices][target_column]

            X_validation_fold = train_df.iloc[validation_indices][feature_columns]
            y_validation_fold = train_df.iloc[validation_indices][target_column]

            estimator = clone(model)
            estimator.fit(X_train_fold, y_train_fold)

            predictions = estimator.predict(X_validation_fold)

            additional_validation_results.append({
                "Model": model_name,
                "Horizon": horizon,
                "Fold": fold_number,
                "MAE": mean_absolute_error(y_validation_fold, predictions),
                "RMSE": np.sqrt(mean_squared_error(y_validation_fold, predictions)),
                "R2": r2_score(y_validation_fold, predictions)
            })

additional_validation_results_df = pd.DataFrame(additional_validation_results)

additional_validation_summary_df = (
    additional_validation_results_df
    .groupby("Model", as_index=False)
    .agg(
        Mean_MAE=("MAE", "mean"),
        Mean_RMSE=("RMSE", "mean"),
        Mean_R2=("R2", "mean")
    )
    .sort_values("Mean_MAE")
    .reset_index(drop=True)
)

print("\nAdditional Machine Learning Validation Performance:")
display(additional_validation_summary_df.round(4))

print("\nTotal Validation Results:", len(additional_validation_results_df))
print("Total Model Fits:", len(additional_validation_results_df))

Evaluating Linear_Regression...
Evaluating Lasso...
Evaluating Decision_Tree...
Evaluating Gradient_Boosting...
Evaluating XGBoost...
Evaluating LightGBM...
Evaluating SVR_RBF...

Additional Machine Learning Validation Performance:


,Model,Mean_MAE,Mean_RMSE,Mean_R2
0,LightGBM,12553.1895,17298.2834,-0.1093
1,XGBoost,12753.0491,17289.5381,-0.1025
2,Lasso,12764.9896,17330.3574,-0.0858
3,Gradient_Boosting,13077.7274,17478.9411,-0.1796
4,Decision_Tree,13281.6096,17627.5868,-0.2405
5,SVR_RBF,13637.2530,19038.0026,-0.2959
6,Linear_Regression,13672.8296,18032.3555,-0.2946



Total Validation Results: 147
Total Model Fits: 147


#### Comparing All Machine Learning Models

In [15]:
all_ml_validation_results_df = pd.concat(
    [
        model_validation_results_df,
        additional_validation_results_df
    ],
    ignore_index=True
)

all_ml_summary_df = (
    all_ml_validation_results_df
    .groupby("Model", as_index=False)
    .agg(
        Mean_MAE=("MAE", "mean"),
        Mean_RMSE=("RMSE", "mean"),
        Mean_R2=("R2", "mean")
    )
    .sort_values("Mean_MAE")
    .reset_index(drop=True)
)

seasonal_baseline_mae = baseline_summary_df.loc[
    baseline_summary_df["Model"] == "Seasonal_Naive",
    "Mean_MAE"
].iloc[0]

all_ml_summary_df["MAE_Improvement_Pct"] = (
    (seasonal_baseline_mae - all_ml_summary_df["Mean_MAE"])
    / seasonal_baseline_mae
    * 100
)

print("Overall Machine Learning Model Comparison:")
display(all_ml_summary_df.round(4))

print("\nTotal Machine Learning Algorithms:", all_ml_summary_df["Model"].nunique())
print("Total Validation Results:", len(all_ml_validation_results_df))
print("Best Model by MAE:", all_ml_summary_df.iloc[0]["Model"])
print("Seasonal Naive Baseline MAE:", round(seasonal_baseline_mae, 2))

Overall Machine Learning Model Comparison:


,Model,Mean_MAE,Mean_RMSE,Mean_R2,MAE_Improvement_Pct
0,Random_Forest,12008.9388,16806.5875,-0.0111,8.9881
1,ElasticNet,12078.3441,16616.4295,0.0322,8.4621
2,Extra_Trees,12269.2499,16949.4003,-0.0539,7.0152
3,Hist_Gradient_Boosting,12460.0799,17179.9529,-0.0751,5.5690
4,LightGBM,12553.1895,17298.2834,-0.1093,4.8634
5,Ridge,12669.5148,17215.4998,-0.0663,3.9818
6,XGBoost,12753.0491,17289.5381,-0.1025,3.3487
7,Lasso,12764.9896,17330.3574,-0.0858,3.2582
8,Gradient_Boosting,13077.7274,17478.9411,-0.1796,0.8881
9,Decision_Tree,13281.6096,17627.5868,-0.2405,-0.6571



Total Machine Learning Algorithms: 12
Total Validation Results: 252
Best Model by MAE: Random_Forest
Seasonal Naive Baseline MAE: 13194.9


**Key Observations**
- **Overall Model Performance:** Random Forest achieved the lowest average MAE (12,008.94), improving upon the Seasonal Naive baseline by 8.99%. ElasticNet followed closely (MAE: 12,078.34) while achieving the lowest RMSE (16,616.43) and highest mean R² (0.0322).
- **Baseline Comparison:** Nine of the twelve machine learning algorithms outperformed Seasonal Naive in average MAE, while Decision Tree, SVR, and Linear Regression performed worse. The generally low or negative R² values highlight the difficulty of predicting revenue fluctuations reliably.

#### Preparing Statistical Forecasting Data

In [16]:
daily_revenue_df = pd.read_parquet(
    data_dir / "daily_revenue.parquet"
)

print("Daily Revenue Dataset:")
print("Shape:", daily_revenue_df.shape)
print("Columns:", daily_revenue_df.columns.tolist())
print("Index Type:", type(daily_revenue_df.index).__name__)

print("\nFirst Five Observations:")
display(daily_revenue_df.head())

print("\nLast Five Observations:")
display(daily_revenue_df.tail())

print("\nForecast Origin Information:")
display(
    train_df[
        ["Forecast_Origin", "Target_Date_H1", "Target_Date_H7"]
    ].head()
)

print("\nTraining Cutoff:", training_end_date.date())
print("Forecast Horizon:", forecast_horizon, "observed trading days")

Daily Revenue Dataset:
Shape: (305, 2)
Columns: ['Transaction_Date', 'Daily_Revenue']
Index Type: RangeIndex

First Five Observations:


,Transaction_Date,Daily_Revenue
0,2010-12-01,57328.60
1,2010-12-02,46174.28
2,2010-12-03,43510.73
3,2010-12-05,30957.28
4,2010-12-06,50907.56



Last Five Observations:


,Transaction_Date,Daily_Revenue
300,2011-12-05,79665.11
301,2011-12-06,52931.35
302,2011-12-07,72694.71
303,2011-12-08,76617.42
304,2011-12-09,29549.88



Forecast Origin Information:


,Forecast_Origin,Target_Date_H1,Target_Date_H7
0,2011-01-16,2011-01-17,2011-01-24
1,2011-01-17,2011-01-18,2011-01-25
2,2011-01-18,2011-01-19,2011-01-26
3,2011-01-19,2011-01-20,2011-01-27
4,2011-01-20,2011-01-21,2011-01-28



Training Cutoff: 2011-09-29
Forecast Horizon: 7 observed trading days


**Key Observations**
- **Time-Series Structure:** The dataset contains 305 chronological revenue observations from December 1, 2010, to December 9, 2011, with `Daily_Revenue` as the forecasting target.
- **Forecasting Alignment:** The seven-step prediction horizon follows observed trading days rather than consecutive calendar days. Statistical models must preserve this ordering and use only revenue observations available at each forecast origin.

#### Preparing the Statistical Forecasting Series

In [17]:
statistical_df = (
    daily_revenue_df
    .copy()
    .assign(
        Transaction_Date=lambda df: pd.to_datetime(df["Transaction_Date"])
    )
    .sort_values("Transaction_Date")
    .reset_index(drop=True)
)

assert statistical_df["Transaction_Date"].is_unique
assert statistical_df["Transaction_Date"].is_monotonic_increasing
assert statistical_df["Daily_Revenue"].notna().all()

statistical_series = statistical_df["Daily_Revenue"].astype(float)

trading_day_positions = pd.Series(
    statistical_df.index.to_numpy(),
    index=statistical_df["Transaction_Date"]
)

validation_origin_positions = []

for fold_number, (_, validation_indices) in enumerate(validation_folds, start=1):
    validation_origins = pd.to_datetime(
        train_df.iloc[validation_indices]["Forecast_Origin"]
    )

    origin_positions = trading_day_positions.loc[
        validation_origins
    ].to_numpy(dtype=int)

    validation_origin_positions.append(origin_positions)

    for origin_position in origin_positions:
        assert origin_position + forecast_horizon < len(statistical_series)

    print(
        f"Fold {fold_number}: "
        f"{len(origin_positions)} forecast origins, "
        f"positions {origin_positions.min()}–{origin_positions.max()}"
    )

for horizon in range(1, forecast_horizon + 1):
    actual_targets = train_df[
        horizon_target_columns[horizon]
    ].to_numpy()

    origin_positions = trading_day_positions.loc[
        pd.to_datetime(train_df["Forecast_Origin"])
    ].to_numpy(dtype=int)

    aligned_targets = statistical_series.iloc[
        origin_positions + horizon
    ].to_numpy()

    np.testing.assert_allclose(
        actual_targets,
        aligned_targets,
        rtol=1e-10,
        atol=1e-6
    )

print("\nStatistical Forecasting Series:")
print("Total Observations:", len(statistical_series))
print("First Date:", statistical_df["Transaction_Date"].min().date())
print("Last Date:", statistical_df["Transaction_Date"].max().date())
print("Target Alignment: Passed for all seven horizons")

Fold 1: 14 forecast origins, positions 195–208
Fold 2: 14 forecast origins, positions 209–222
Fold 3: 14 forecast origins, positions 223–236

Statistical Forecasting Series:
Total Observations: 305
First Date: 2010-12-01
Last Date: 2011-12-09
Target Alignment: Passed for all seven horizons


**Key Observations**
- **Validation Alignment:** All three expanding-window validation folds contain 14 forecast origins, covering trading-day positions 195–236 without disrupting chronological ordering.
- **Target Consistency:** Actual revenue values match the supervised forecasting targets across all seven horizons, ensuring that statistical and machine learning models can be evaluated against identical historical outcomes.

#### Configuring Statistical Forecasting Models

In [18]:
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing

statistical_model_config = {
    "AutoReg": {
        "lags": [1, 2, 3, 6, 12]
    },
    "ARIMA": {
        "order": (2, 0, 2)
    },
    "SARIMA": {
        "order": (1, 0, 1),
        "seasonal_order": (1, 0, 1, 6)
    },
    "SARIMAX": {
        "order": (1, 0, 1),
        "seasonal_order": (1, 0, 1, 6),
        "exogenous_feature": "Revenue_Lag_6"
    },
    "ETS": {
        "trend": "add",
        "seasonal": "add",
        "seasonal_periods": 6
    }
}

statistical_configuration_df = pd.DataFrame([
    {
        "Model": model_name,
        "Configuration": str(parameters)
    }
    for model_name, parameters in statistical_model_config.items()
])

print("Configured Statistical Forecasting Models:")
display(statistical_configuration_df)

print("\nTotal Statistical Models:", len(statistical_model_config))
print("Validation Forecast Origins:", sum(len(positions) for positions in validation_origin_positions))
print("Forecast Horizons:", forecast_horizon)
print(
    "Estimated Model Fits:",
    len(statistical_model_config)
    * sum(len(positions) for positions in validation_origin_positions)
)

Configured Statistical Forecasting Models:


,Model,Configuration
0,AutoReg,"{'lags': [1, 2, 3, 6, 12]}"
1,ARIMA,"{'order': (2, 0, 2)}"
2,SARIMA,"{'order': (1, 0, 1), 'seasonal_order': (1, 0, ..."
3,SARIMAX,"{'order': (1, 0, 1), 'seasonal_order': (1, 0, ..."
4,ETS,"{'trend': 'add', 'seasonal': 'add', 'seasonal_..."



Total Statistical Models: 5
Validation Forecast Origins: 42
Forecast Horizons: 7
Estimated Model Fits: 210


**Key Observations**
- **Statistical Model Coverage:** Five forecasting approaches have been configured, including autoregressive, seasonal, exogenous, and exponential smoothing methods, with a six-observed-trading-day seasonal period.
- **Validation Framework:** The statistical models will be evaluated across 42 historical forecast origins and seven prediction horizons, requiring approximately 210 model fits while preserving chronological information boundaries.

#### Preparing Leakage-Safe Exogenous Forecasting Features

In [19]:
statistical_exog_df = statistical_df[
    ["Transaction_Date"]
].copy()

statistical_exog_df["Day_Of_Week"] = (
    statistical_exog_df["Transaction_Date"].dt.dayofweek
)

statistical_exog_df["Month"] = (
    statistical_exog_df["Transaction_Date"].dt.month
)

statistical_exog_df["Is_Month_End"] = (
    statistical_exog_df["Transaction_Date"].dt.is_month_end.astype(int)
)

statistical_exog_df["Day_Of_Week_Sin"] = np.sin(
    2 * np.pi * statistical_exog_df["Day_Of_Week"] / 7
)

statistical_exog_df["Day_Of_Week_Cos"] = np.cos(
    2 * np.pi * statistical_exog_df["Day_Of_Week"] / 7
)

statistical_exog_df["Month_Sin"] = np.sin(
    2 * np.pi * (statistical_exog_df["Month"] - 1) / 12
)

statistical_exog_df["Month_Cos"] = np.cos(
    2 * np.pi * (statistical_exog_df["Month"] - 1) / 12
)

statistical_exog_columns = [
    "Day_Of_Week_Sin",
    "Day_Of_Week_Cos",
    "Month_Sin",
    "Month_Cos",
    "Is_Month_End"
]

statistical_exog = statistical_exog_df[
    statistical_exog_columns
].astype(float)

statistical_model_config["SARIMAX"]["exogenous_feature"] = (
    statistical_exog_columns
)

assert statistical_exog.notna().all().all()
assert len(statistical_exog) == len(statistical_series)

for origin_positions in validation_origin_positions:
    for origin_position in origin_positions:
        future_positions = np.arange(
            origin_position + 1,
            origin_position + forecast_horizon + 1
        )

        assert len(
            statistical_exog.iloc[future_positions]
        ) == forecast_horizon

print("SARIMAX Exogenous Features:")
print(statistical_exog_columns)

print("\nExogenous Feature Matrix Shape:", statistical_exog.shape)
print("Missing Values:", statistical_exog.isna().sum().sum())
print("Future Feature Alignment: Passed")

display(statistical_exog.head())

SARIMAX Exogenous Features:
['Day_Of_Week_Sin', 'Day_Of_Week_Cos', 'Month_Sin', 'Month_Cos', 'Is_Month_End']

Exogenous Feature Matrix Shape: (305, 5)
Missing Values: 0
Future Feature Alignment: Passed


C:\Users\user\AppData\Local\Temp\ipykernel_5108\1749948276.py:6: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  statistical_exog_df["Transaction_Date"].dt.dayofweek


,Day_Of_Week_Sin,Day_Of_Week_Cos,Month_Sin,Month_Cos,Is_Month_End
0,0.974928,-0.222521,-0.5,0.866025,0.0
1,0.433884,-0.900969,-0.5,0.866025,0.0
2,-0.433884,-0.900969,-0.5,0.866025,0.0
3,-0.781831,0.623490,-0.5,0.866025,0.0
4,0.000000,1.000000,-0.5,0.866025,0.0


**Key Observations**
- **Exogenous Feature Preparation:** Five calendar-derived features were generated across all 305 trading-day observations, capturing weekly patterns, monthly seasonality, and month-end effects without missing values.
- **Forecasting Integrity:** Future exogenous features align with all seven prediction horizons and do not depend on future revenue observations, preserving target leakage prevention under the known trading-calendar assumption.

#### Evaluating Statistical Forecasting Models

In [20]:
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning

statistical_validation_results = []
statistical_fit_issues = []

for model_name, parameters in statistical_model_config.items():
    print(f"Evaluating {model_name}...")

    for fold_number, origin_positions in enumerate(
        validation_origin_positions,
        start=1
    ):
        for origin_position in origin_positions:
            historical_revenue = statistical_series.iloc[
                :origin_position + 1
            ].reset_index(drop=True)

            actual_values = statistical_series.iloc[
                origin_position + 1:
                origin_position + forecast_horizon + 1
            ].to_numpy()

            try:
                with warnings.catch_warnings(record=True) as caught_warnings:
                    warnings.simplefilter("always", ConvergenceWarning)

                    if model_name == "AutoReg":
                        fitted_model = AutoReg(
                            historical_revenue,
                            lags=parameters["lags"],
                            old_names=False
                        ).fit()

                        predictions = fitted_model.forecast(
                            steps=forecast_horizon
                        )

                    elif model_name == "ARIMA":
                        fitted_model = ARIMA(
                            historical_revenue,
                            order=parameters["order"]
                        ).fit()

                        predictions = fitted_model.forecast(
                            steps=forecast_horizon
                        )

                    elif model_name == "SARIMA":
                        fitted_model = SARIMAX(
                            historical_revenue,
                            order=parameters["order"],
                            seasonal_order=parameters["seasonal_order"],
                            enforce_stationarity=False,
                            enforce_invertibility=False
                        ).fit(disp=False, maxiter=100)

                        predictions = fitted_model.forecast(
                            steps=forecast_horizon
                        )

                    elif model_name == "SARIMAX":
                        historical_exog = statistical_exog.iloc[
                            :origin_position + 1
                        ].reset_index(drop=True)

                        future_exog = statistical_exog.iloc[
                            origin_position + 1:
                            origin_position + forecast_horizon + 1
                        ].reset_index(drop=True)

                        fitted_model = SARIMAX(
                            historical_revenue,
                            exog=historical_exog,
                            order=parameters["order"],
                            seasonal_order=parameters["seasonal_order"],
                            enforce_stationarity=False,
                            enforce_invertibility=False
                        ).fit(disp=False, maxiter=100)

                        predictions = fitted_model.forecast(
                            steps=forecast_horizon,
                            exog=future_exog
                        )

                    elif model_name == "ETS":
                        fitted_model = ExponentialSmoothing(
                            historical_revenue,
                            trend=parameters["trend"],
                            seasonal=parameters["seasonal"],
                            seasonal_periods=parameters["seasonal_periods"],
                            initialization_method="estimated"
                        ).fit(optimized=True)

                        predictions = fitted_model.forecast(
                            steps=forecast_horizon
                        )

                predictions = np.asarray(predictions, dtype=float)

                if len(predictions) != forecast_horizon or not np.isfinite(predictions).all():
                    raise ValueError("Invalid or non-finite forecast values.")

                for horizon, (actual, predicted) in enumerate(
                    zip(actual_values, predictions),
                    start=1
                ):
                    statistical_validation_results.append({
                        "Model": model_name,
                        "Horizon": horizon,
                        "Fold": fold_number,
                        "Forecast_Origin": statistical_df.loc[
                            origin_position, "Transaction_Date"
                        ],
                        "Actual": actual,
                        "Predicted": predicted,
                        "Absolute_Error": abs(actual - predicted),
                        "Squared_Error": (actual - predicted) ** 2
                    })

                for warning_record in caught_warnings:
                    if issubclass(warning_record.category, ConvergenceWarning):
                        statistical_fit_issues.append({
                            "Model": model_name,
                            "Fold": fold_number,
                            "Origin_Position": origin_position,
                            "Issue": "Convergence warning"
                        })

            except Exception as error:
                statistical_fit_issues.append({
                    "Model": model_name,
                    "Fold": fold_number,
                    "Origin_Position": origin_position,
                    "Issue": str(error)
                })

statistical_validation_results_df = pd.DataFrame(statistical_validation_results)
statistical_fit_issues_df = pd.DataFrame(statistical_fit_issues)

print("\nStatistical Forecasting Evaluation:")
print("Successful Horizon Predictions:", len(statistical_validation_results_df))
print("Expected Horizon Predictions:", 5 * 42 * forecast_horizon)
print("Recorded Fit Issues:", len(statistical_fit_issues_df))

if not statistical_validation_results_df.empty:
    statistical_summary_df = (
        statistical_validation_results_df
        .groupby("Model")
        .agg(
            Mean_MAE=("Absolute_Error", "mean"),
            Mean_RMSE=("Squared_Error", lambda x: np.sqrt(x.mean())),
            Forecast_Count=("Actual", "count")
        )
        .sort_values("Mean_MAE")
        .reset_index()
    )

    display(statistical_summary_df.round(4))

if not statistical_fit_issues_df.empty:
    print("\nFit Issues by Model:")
    display(
        statistical_fit_issues_df.groupby(
            ["Model", "Issue"]
        ).size().reset_index(name="Count")
    )

Evaluating AutoReg...
Evaluating ARIMA...
Evaluating SARIMA...
Evaluating SARIMAX...
Evaluating ETS...

Statistical Forecasting Evaluation:
Successful Horizon Predictions: 1176
Expected Horizon Predictions: 1470
Recorded Fit Issues: 43


,Model,Mean_MAE,Mean_RMSE,Forecast_Count
0,SARIMA,11980.8481,18085.2750,294
1,ETS,12189.1787,17921.9726,294
2,ARIMA,12862.5776,19339.8242,294
3,SARIMAX,15684.5767,21138.4057,294



Fit Issues by Model:


,Model,Issue,Count
0,ARIMA,Convergence warning,1
1,AutoReg,AutoReg.__init__() got an unexpected keyword a...,42


**Key Observations**
- **Statistical Forecasting Performance:** SARIMA achieved the lowest average MAE (11,980.85), followed by ETS (12,189.18), while SARIMAX recorded the highest prediction errors among the successfully evaluated models.
- **Model Execution:** Four statistical models generated all 294 expected horizon predictions each. AutoReg failed at all 42 forecast origins because of an unsupported constructor argument, while ARIMA produced one convergence warning.

#### Correcting and Reevaluating AutoReg

In [21]:
autoreg_validation_results = []
autoreg_fit_issues = []

for fold_number, origin_positions in enumerate(
    validation_origin_positions,
    start=1
):
    for origin_position in origin_positions:
        historical_revenue = statistical_series.iloc[
            :origin_position + 1
        ].reset_index(drop=True)

        actual_values = statistical_series.iloc[
            origin_position + 1:
            origin_position + forecast_horizon + 1
        ].to_numpy()

        try:
            fitted_model = AutoReg(
                historical_revenue,
                lags=statistical_model_config["AutoReg"]["lags"]
            ).fit()

            predictions = np.asarray(
                fitted_model.forecast(steps=forecast_horizon),
                dtype=float
            )

            if len(predictions) != forecast_horizon or not np.isfinite(predictions).all():
                raise ValueError("Invalid or non-finite forecast values.")

            for horizon, (actual, predicted) in enumerate(
                zip(actual_values, predictions),
                start=1
            ):
                autoreg_validation_results.append({
                    "Model": "AutoReg",
                    "Horizon": horizon,
                    "Fold": fold_number,
                    "Forecast_Origin": statistical_df.loc[
                        origin_position, "Transaction_Date"
                    ],
                    "Actual": actual,
                    "Predicted": predicted,
                    "Absolute_Error": abs(actual - predicted),
                    "Squared_Error": (actual - predicted) ** 2
                })

        except Exception as error:
            autoreg_fit_issues.append({
                "Model": "AutoReg",
                "Fold": fold_number,
                "Origin_Position": origin_position,
                "Issue": str(error)
            })

autoreg_validation_results_df = pd.DataFrame(autoreg_validation_results)

statistical_validation_results_df = pd.concat(
    [
        statistical_validation_results_df[
            statistical_validation_results_df["Model"] != "AutoReg"
        ],
        autoreg_validation_results_df
    ],
    ignore_index=True
)

statistical_fit_issues_df = pd.concat(
    [
        statistical_fit_issues_df[
            statistical_fit_issues_df["Model"] != "AutoReg"
        ],
        pd.DataFrame(autoreg_fit_issues)
    ],
    ignore_index=True
)

print("AutoReg Successful Predictions:", len(autoreg_validation_results_df))
print("AutoReg Expected Predictions:", 42 * forecast_horizon)
print("AutoReg Fit Failures:", len(autoreg_fit_issues))

print("\nUpdated Statistical Forecasting Results:")
print("Total Successful Predictions:", len(statistical_validation_results_df))
print("Total Expected Predictions:", 5 * 42 * forecast_horizon)

statistical_summary_df = (
    statistical_validation_results_df
    .groupby("Model")
    .agg(
        Mean_MAE=("Absolute_Error", "mean"),
        Mean_RMSE=("Squared_Error", lambda x: np.sqrt(x.mean())),
        Forecast_Count=("Actual", "count")
    )
    .sort_values("Mean_MAE")
    .reset_index()
)

display(statistical_summary_df.round(4))

if autoreg_fit_issues:
    print("\nAutoReg Fit Issues:")
    display(pd.DataFrame(autoreg_fit_issues))

AutoReg Successful Predictions: 294
AutoReg Expected Predictions: 294
AutoReg Fit Failures: 0

Updated Statistical Forecasting Results:
Total Successful Predictions: 1470
Total Expected Predictions: 1470


,Model,Mean_MAE,Mean_RMSE,Forecast_Count
0,AutoReg,11812.5487,18049.6545,294
1,SARIMA,11980.8481,18085.2750,294
2,ETS,12189.1787,17921.9726,294
3,ARIMA,12862.5776,19339.8242,294
4,SARIMAX,15684.5767,21138.4057,294


**Key Observations**
- **Statistical Model Performance:** AutoReg achieved the lowest average MAE (11,812.55), followed by SARIMA (11,980.85) and ETS (12,189.18). ETS recorded the lowest pooled RMSE (17,921.97), while SARIMAX performed weakest.
- **Evaluation Completion:** All five statistical models successfully generated 294 predictions each, completing the full set of 1,470 horizon-level predictions across 42 validation origins.

#### Standardizing Statistical Model Evaluation Metrics

In [22]:
statistical_fold_metrics = []

for (model_name, horizon, fold_number), group in (
    statistical_validation_results_df.groupby(
        ["Model", "Horizon", "Fold"]
    )
):
    actual = group["Actual"].to_numpy()
    predicted = group["Predicted"].to_numpy()

    statistical_fold_metrics.append({
        "Model": model_name,
        "Horizon": horizon,
        "Fold": fold_number,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": r2_score(actual, predicted)
    })

statistical_validation_metrics_df = pd.DataFrame(
    statistical_fold_metrics
)

statistical_standardized_summary_df = (
    statistical_validation_metrics_df
    .groupby("Model", as_index=False)
    .agg(
        Mean_MAE=("MAE", "mean"),
        Mean_RMSE=("RMSE", "mean"),
        Mean_R2=("R2", "mean")
    )
    .sort_values("Mean_MAE")
    .reset_index(drop=True)
)

expected_metric_rows = (
    len(statistical_model_config)
    * forecast_horizon
    * len(validation_folds)
)

assert len(statistical_validation_metrics_df) == expected_metric_rows
assert (
    statistical_validation_metrics_df.groupby(
        ["Model", "Horizon", "Fold"]
    ).size() == 1
).all()

print("Standardized Statistical Model Performance:")
display(statistical_standardized_summary_df.round(4))

print("\nTotal Statistical Models:", statistical_standardized_summary_df["Model"].nunique())
print("Total Fold-Horizon Metric Rows:", len(statistical_validation_metrics_df))
print("Expected Metric Rows:", expected_metric_rows)

print("\nBest Statistical Model by MAE:")
print(statistical_standardized_summary_df.iloc[0]["Model"])

Standardized Statistical Model Performance:


,Model,Mean_MAE,Mean_RMSE,Mean_R2
0,AutoReg,11812.5487,16419.5086,0.0166
1,SARIMA,11980.8481,16715.9763,-0.1623
2,ETS,12189.1787,16657.1399,-0.1456
3,ARIMA,12862.5776,17981.0859,-0.1505
4,SARIMAX,15684.5767,20434.8020,-1.1531



Total Statistical Models: 5
Total Fold-Horizon Metric Rows: 105
Expected Metric Rows: 105

Best Statistical Model by MAE:
AutoReg


                                              0.0/100.2 MB ? eta -:--:--
                                              0.0/100.2 MB 1.9 MB/s eta 0:00:53
                                              0.0/100.2 MB 1.9 MB/s eta 0:00:53
                                              0.0/100.2 MB 1.9 MB/s eta 0:00:53
                                              0.0/100.2 MB 1.9 MB/s eta 0:00:53
                                              0.0/100.2 MB 1.9 MB/s eta 0:00:53
                                              0.0/100.2 MB 1.9 MB/s eta 0:00:53
                                              0.0/100.2 MB 1.9 MB/s eta 0:00:53
                                              0.0/100.2 MB 1.9 MB/s eta 0:00:53
                                            0.1/100.2 MB 235.7 kB/s eta 0:07:05
                                            0.1/100.2 MB 235.7 kB/s eta 0:07:05
                                            0.1/100.2 MB 225.3 kB/s eta 0:07:25
                                            0.1


[notice] A new release of pip is available: 23.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



     -----------------------               62.5/100.2 MB 229.3 kB/s eta 0:02:45
     -----------------------               62.5/100.2 MB 229.3 kB/s eta 0:02:45
     -----------------------               62.5/100.2 MB 229.3 kB/s eta 0:02:45
     -----------------------               62.5/100.2 MB 229.3 kB/s eta 0:02:45
     -----------------------               62.5/100.2 MB 229.3 kB/s eta 0:02:45
     -----------------------               62.5/100.2 MB 229.3 kB/s eta 0:02:45
     -----------------------               62.5/100.2 MB 229.3 kB/s eta 0:02:45
     -----------------------               62.5/100.2 MB 222.9 kB/s eta 0:02:50
     -----------------------               62.5/100.2 MB 222.9 kB/s eta 0:02:50
     -----------------------               62.5/100.2 MB 222.9 kB/s eta 0:02:50
     -----------------------               62.5/100.2 MB 222.9 kB/s eta 0:02:50
     -----------------------               62.5/100.2 MB 222.9 kB/s eta 0:02:50
     -----------------------           

**Key Observations**
- **Statistical Forecasting Performance:** AutoReg achieved the lowest average MAE (11,812.55) and RMSE (16,419.51), outperforming SARIMA and ETS while maintaining a slightly positive mean R² (0.0166).
- **Evaluation Consistency:** All five statistical models were evaluated using 105 fold-horizon metric combinations, establishing a consistent basis for comparison with the previously evaluated machine learning algorithms.

#### Comparing Machine Learning and Statistical Forecasting Models

In [23]:
ml_comparison_df = all_ml_summary_df[
    ["Model", "Mean_MAE", "Mean_RMSE", "Mean_R2"]
].copy()

ml_comparison_df["Model_Family"] = "Machine Learning"

statistical_comparison_df = statistical_standardized_summary_df[
    ["Model", "Mean_MAE", "Mean_RMSE", "Mean_R2"]
].copy()

statistical_comparison_df["Model_Family"] = "Statistical"

combined_model_comparison_df = pd.concat(
    [
        ml_comparison_df,
        statistical_comparison_df
    ],
    ignore_index=True
)

combined_model_comparison_df["MAE_Improvement_Pct"] = (
    (
        seasonal_baseline_mae
        - combined_model_comparison_df["Mean_MAE"]
    )
    / seasonal_baseline_mae
    * 100
)

combined_model_comparison_df = (
    combined_model_comparison_df
    .sort_values("Mean_MAE")
    .reset_index(drop=True)
)

print("Machine Learning and Statistical Model Comparison:")
display(combined_model_comparison_df.round(4))

print("\nTotal Models:", len(combined_model_comparison_df))
print(
    "Models Outperforming Seasonal Naive:",
    (combined_model_comparison_df["MAE_Improvement_Pct"] > 0).sum()
)

print("\nTop Five Models:")
display(combined_model_comparison_df.head(5).round(4))

print("\nBest Overall Model by MAE:")
print(combined_model_comparison_df.iloc[0]["Model"])

Machine Learning and Statistical Model Comparison:


,Model,Mean_MAE,Mean_RMSE,Mean_R2,Model_Family,MAE_Improvement_Pct
0,AutoReg,11812.5487,16419.5086,0.0166,Statistical,10.4764
1,SARIMA,11980.8481,16715.9763,-0.1623,Statistical,9.2010
2,Random_Forest,12008.9388,16806.5875,-0.0111,Machine Learning,8.9881
3,ElasticNet,12078.3441,16616.4295,0.0322,Machine Learning,8.4621
4,ETS,12189.1787,16657.1399,-0.1456,Statistical,7.6221
5,Extra_Trees,12269.2499,16949.4003,-0.0539,Machine Learning,7.0152
6,Hist_Gradient_Boosting,12460.0799,17179.9529,-0.0751,Machine Learning,5.5690
7,LightGBM,12553.1895,17298.2834,-0.1093,Machine Learning,4.8634
8,Ridge,12669.5148,17215.4998,-0.0663,Machine Learning,3.9818
9,XGBoost,12753.0491,17289.5381,-0.1025,Machine Learning,3.3487



Total Models: 17
Models Outperforming Seasonal Naive: 13

Top Five Models:


,Model,Mean_MAE,Mean_RMSE,Mean_R2,Model_Family,MAE_Improvement_Pct
0,AutoReg,11812.5487,16419.5086,0.0166,Statistical,10.4764
1,SARIMA,11980.8481,16715.9763,-0.1623,Statistical,9.2010
2,Random_Forest,12008.9388,16806.5875,-0.0111,Machine Learning,8.9881
3,ElasticNet,12078.3441,16616.4295,0.0322,Machine Learning,8.4621
4,ETS,12189.1787,16657.1399,-0.1456,Statistical,7.6221



Best Overall Model by MAE:
AutoReg


**Key Observations**
- **Overall Forecasting Performance:** AutoReg achieved the lowest average MAE (11,812.55), outperforming the Seasonal Naive baseline by 10.48%. SARIMA, Random Forest, and ElasticNet followed closely, demonstrating competitive performance across statistical and machine learning approaches.
- **Model Comparison:** Thirteen of the seventeen models improved upon the Seasonal Naive baseline, while ElasticNet achieved the highest mean R² (0.0322). The relatively modest improvements and generally low R² values highlight the challenges of forecasting daily retail revenue.

#### Checking Neural Network Dependencies

In [25]:
import importlib.util
import sys

neural_network_libraries = {
    "TensorFlow": "tensorflow",
    "PyTorch": "torch",
    "Scikit-learn": "sklearn"
}

dependency_results = []

for library_name, module_name in neural_network_libraries.items():
    installed = importlib.util.find_spec(module_name) is not None

    dependency_results.append({
        "Library": library_name,
        "Installed": installed
    })

dependency_df = pd.DataFrame(dependency_results)

print("Neural Network Dependencies:")
display(dependency_df)

print("\nPython Version:", sys.version.split()[0])
print("Available CPU Cores:", __import__("os").cpu_count())

Neural Network Dependencies:


,Library,Installed
0,TensorFlow,False
1,PyTorch,True
2,Scikit-learn,True



Python Version: 3.11.4
Available CPU Cores: 8


#### Configuring Neural Network Training Environment

In [26]:
import random
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

random_seed = 42

random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)

torch.set_num_threads(4)

device = torch.device("cpu")

neural_network_config = {
    "random_seed": random_seed,
    "device": str(device),
    "cpu_threads": torch.get_num_threads(),
    "forecast_horizon": forecast_horizon,
    "batch_size": 16,
    "max_epochs": 100,
    "patience": 12,
    "learning_rate": 0.001
}

print("Neural Network Training Configuration:")
display(pd.DataFrame(
    neural_network_config.items(),
    columns=["Parameter", "Value"]
))

print("\nPyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("Training Device:", device)

Neural Network Training Configuration:


,Parameter,Value
0,random_seed,42
1,device,cpu
2,cpu_threads,4
3,forecast_horizon,7
4,batch_size,16
5,max_epochs,100
6,patience,12
7,learning_rate,0.001



PyTorch Version: 2.15.0+cpu
CUDA Available: False
Training Device: cpu


**Key Observations**
- **Training Environment:** PyTorch 2.15.0+cpu is configured for CPU-based neural network training with four threads and a fixed random seed of 42.
- **Training Configuration:** A seven-day forecasting horizon, batch size of 16, maximum of 100 epochs, and early stopping patience of 12 have been established to support efficient experimentation and reduce overfitting.

#### Preparing Neural Network Input Sequences

In [27]:
sequence_length = 24

forecast_origins = pd.to_datetime(
    train_df["Forecast_Origin"]
)

origin_positions = trading_day_positions.loc[
    forecast_origins
].to_numpy(dtype=int)

neural_X = []
neural_y = []

for origin_position in origin_positions:
    sequence_start = origin_position - sequence_length + 1
    target_end = origin_position + forecast_horizon + 1

    assert sequence_start >= 0
    assert target_end <= len(statistical_series)

    input_sequence = statistical_series.iloc[
        sequence_start:origin_position + 1
    ].to_numpy(dtype=np.float32)

    target_sequence = statistical_series.iloc[
        origin_position + 1:target_end
    ].to_numpy(dtype=np.float32)

    neural_X.append(input_sequence)
    neural_y.append(target_sequence)

neural_X = np.asarray(neural_X, dtype=np.float32)
neural_y = np.asarray(neural_y, dtype=np.float32)

assert neural_X.shape == (
    len(train_df),
    sequence_length
)

assert neural_y.shape == (
    len(train_df),
    forecast_horizon
)

for horizon in range(1, forecast_horizon + 1):
    np.testing.assert_allclose(
        neural_y[:, horizon - 1],
        train_df[
            horizon_target_columns[horizon]
        ].to_numpy(),
        rtol=1e-6,
        atol=0.01
    )

print("Neural Network Sequence Preparation:")
print("Input Shape:", neural_X.shape)
print("Target Shape:", neural_y.shape)
print("Sequence Length:", sequence_length)
print("Forecast Horizon:", forecast_horizon)
print("Total Forecast Origins:", len(origin_positions))
print("Target Alignment: Passed")

print("\nFirst Input Sequence:")
print(neural_X[0])

print("\nFirst Target Sequence:")
print(neural_y[0])

print("\nValidation Fold Sizes:")
for fold_number, (train_indices, validation_indices) in enumerate(
    validation_folds,
    start=1
):
    print(
        f"Fold {fold_number}: "
        f"Training={len(train_indices)}, "
        f"Validation={len(validation_indices)}"
    )

Neural Network Sequence Preparation:
Input Shape: (207, 24)
Target Shape: (207, 7)
Sequence Length: 24
Forecast Horizon: 7
Total Forecast Origins: 207
Target Alignment: Passed

First Input Sequence:
[48978.46 55205.1  17240.92 36425.76 42007.09 29517.91 48088.35 39404.72
  7395.31 23504.21 43469.76  4791.17 10989.24 14309.63 31179.72 38712.02
 26058.22 15520.8  22943.86 67059.54 22719.81 19519.07 45977.64  7114.11]

First Target Sequence:
[27771.28 17097.6  24801.64 20062.27 30886.09 10177.95 24358.  ]

Validation Fold Sizes:
Fold 1: Training=159, Validation=14
Fold 2: Training=173, Validation=14
Fold 3: Training=187, Validation=14


**Key Observations**
- **Sequence Preparation:** A total of 207 input sequences were generated, each containing 24 historical trading-day revenue observations and seven future revenue targets, with target alignment successfully verified.
- **Validation Consistency:** The existing three expanding-window folds were preserved, containing 159, 173, and 187 training samples respectively, with 14 validation samples per fold.

#### Configuring Neural Network Data Preprocessing

In [28]:
def prepare_neural_fold(
    train_indices,
    validation_indices,
    validation_fraction=0.15
):
    train_indices = np.asarray(train_indices)
    validation_indices = np.asarray(validation_indices)

    early_stopping_size = max(
        10,
        int(np.ceil(len(train_indices) * validation_fraction))
    )

    early_stopping_indices = train_indices[-early_stopping_size:]

    early_stopping_start_position = origin_positions[
        early_stopping_indices[0]
    ]

    fitting_indices = train_indices[
        origin_positions[train_indices] + forecast_horizon
        <= early_stopping_start_position
    ]

    if len(fitting_indices) < 30:
        raise ValueError(
            "Insufficient training samples after chronological purging."
        )

    X_scaler = StandardScaler()
    y_scaler = StandardScaler()

    X_fit = X_scaler.fit_transform(
        neural_X[fitting_indices]
    )

    y_fit = y_scaler.fit_transform(
        neural_y[fitting_indices]
    )

    X_early_stop = X_scaler.transform(
        neural_X[early_stopping_indices]
    )

    y_early_stop = y_scaler.transform(
        neural_y[early_stopping_indices]
    )

    X_validation = X_scaler.transform(
        neural_X[validation_indices]
    )

    y_validation = neural_y[validation_indices]

    return {
        "X_fit": X_fit.astype(np.float32),
        "y_fit": y_fit.astype(np.float32),
        "X_early_stop": X_early_stop.astype(np.float32),
        "y_early_stop": y_early_stop.astype(np.float32),
        "X_validation": X_validation.astype(np.float32),
        "y_validation": y_validation.astype(np.float32),
        "X_scaler": X_scaler,
        "y_scaler": y_scaler,
        "fitting_indices": fitting_indices,
        "early_stopping_indices": early_stopping_indices
    }

neural_fold_preparation_summary = []

for fold_number, (train_indices, validation_indices) in enumerate(
    validation_folds,
    start=1
):
    fold_data = prepare_neural_fold(
        train_indices,
        validation_indices
    )

    assert (
        origin_positions[fold_data["fitting_indices"][-1]]
        + forecast_horizon
        <= origin_positions[fold_data["early_stopping_indices"][0]]
    )

    assert np.isfinite(fold_data["X_fit"]).all()
    assert np.isfinite(fold_data["y_fit"]).all()

    neural_fold_preparation_summary.append({
        "Fold": fold_number,
        "Fitting_Samples": len(fold_data["fitting_indices"]),
        "Early_Stopping_Samples": len(fold_data["early_stopping_indices"]),
        "Validation_Samples": len(validation_indices),
        "Input_Features": fold_data["X_fit"].shape[1],
        "Output_Targets": fold_data["y_fit"].shape[1]
    })

print("Neural Network Fold Preparation:")
display(pd.DataFrame(neural_fold_preparation_summary))

print("\nChronological Purging: Passed")
print("Training-Only Scaling: Configured")
print("Input and Target Validation: Passed")

Neural Network Fold Preparation:


,Fold,Fitting_Samples,Early_Stopping_Samples,Validation_Samples,Input_Features,Output_Targets
0,1,129,24,14,24,7
1,2,141,26,14,24,7
2,3,152,29,14,24,7



Chronological Purging: Passed
Training-Only Scaling: Configured
Input and Target Validation: Passed


**Key Observations**
- **Training Data Preparation:** The three validation folds contain 129, 141, and 152 fitting samples, with separate chronological early-stopping subsets of 24, 26, and 29 samples respectively.
- **Leakage Prevention:** Chronological purging successfully separates fitting and early-stopping target periods, while training-only scaling preserves validation independence across all 24 input features and seven forecasting targets.

#### Defining the Multilayer Perceptron Architecture

In [29]:
class MLPForecaster(nn.Module):
    def __init__(
        self,
        input_size=24,
        hidden_size=32,
        output_size=7,
        dropout=0.2
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Linear(hidden_size // 2, output_size)
        )

    def forward(self, x):
        return self.network(x)

mlp_model = MLPForecaster(
    input_size=sequence_length,
    hidden_size=32,
    output_size=forecast_horizon,
    dropout=0.2
).to(device)

sample_input = torch.tensor(
    neural_X[:2],
    dtype=torch.float32,
    device=device
)

with torch.no_grad():
    sample_output = mlp_model(sample_input)

total_parameters = sum(
    parameter.numel()
    for parameter in mlp_model.parameters()
    if parameter.requires_grad
)

print("MLP Architecture:")
print(mlp_model)

print("\nInput Shape:", tuple(sample_input.shape))
print("Output Shape:", tuple(sample_output.shape))
print("Trainable Parameters:", total_parameters)
print("Training Device:", device)

assert sample_output.shape == (2, forecast_horizon)

print("\nArchitecture Validation: Passed")

MLP Architecture:
MLPForecaster(
  (network): Sequential(
    (0): Linear(in_features=24, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=32, out_features=16, bias=True)
    (4): ReLU()
    (5): Linear(in_features=16, out_features=7, bias=True)
  )
)

Input Shape: (2, 24)
Output Shape: (2, 7)
Trainable Parameters: 1447
Training Device: cpu

Architecture Validation: Passed


**Key Observations**
- **Network Architecture:** The MLP consists of two hidden layers containing 32 and 16 neurons, with ReLU activation and 20% dropout regularization to reduce overfitting.
- **Architecture Validation:** The model contains 1,447 trainable parameters and successfully transforms 24 historical revenue inputs into seven forecast outputs using the CPU.

#### Configuring Neural Network Training and Early Stopping

In [30]:
import copy

def train_neural_network(model, fold_data, config):
    X_fit = torch.tensor(
        fold_data["X_fit"],
        dtype=torch.float32
    )
    y_fit = torch.tensor(
        fold_data["y_fit"],
        dtype=torch.float32
    )

    X_early_stop = torch.tensor(
        fold_data["X_early_stop"],
        dtype=torch.float32,
        device=device
    )
    y_early_stop = torch.tensor(
        fold_data["y_early_stop"],
        dtype=torch.float32,
        device=device
    )

    training_dataset = TensorDataset(X_fit, y_fit)

    training_loader = DataLoader(
        training_dataset,
        batch_size=config["batch_size"],
        shuffle=True
    )

    model = model.to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["learning_rate"]
    )

    criterion = nn.MSELoss()

    best_validation_loss = float("inf")
    best_model_state = None
    epochs_without_improvement = 0

    training_history = []

    for epoch in range(1, config["max_epochs"] + 1):
        model.train()
        epoch_training_loss = 0.0

        for X_batch, y_batch in training_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)

            loss.backward()
            optimizer.step()

            epoch_training_loss += (
                loss.item() * len(X_batch)
            )

        epoch_training_loss /= len(training_dataset)

        model.eval()

        with torch.no_grad():
            early_stop_predictions = model(X_early_stop)

            validation_loss = criterion(
                early_stop_predictions,
                y_early_stop
            ).item()

        training_history.append({
            "Epoch": epoch,
            "Training_Loss": epoch_training_loss,
            "Validation_Loss": validation_loss
        })

        if validation_loss < best_validation_loss - 1e-6:
            best_validation_loss = validation_loss
            best_model_state = copy.deepcopy(
                model.state_dict()
            )
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= config["patience"]:
            break

    model.load_state_dict(best_model_state)
    model.eval()

    history_df = pd.DataFrame(training_history)

    return {
        "model": model,
        "history": history_df,
        "best_epoch": int(
            history_df.loc[
                history_df["Validation_Loss"].idxmin(),
                "Epoch"
            ]
        ),
        "best_validation_loss": best_validation_loss,
        "epochs_trained": len(history_df)
    }

print("Neural Network Training Function: Configured")
print("Optimizer: Adam")
print("Loss Function: Mean Squared Error")
print("Early Stopping Patience:", neural_network_config["patience"])
print("Maximum Epochs:", neural_network_config["max_epochs"])
print("Best Model Weight Restoration: Enabled")

Neural Network Training Function: Configured
Optimizer: Adam
Loss Function: Mean Squared Error
Early Stopping Patience: 12
Maximum Epochs: 100
Best Model Weight Restoration: Enabled


**Key Observations**
- **Training Configuration:** A reusable PyTorch training function has been configured with Adam optimization, Mean Squared Error loss, and mini-batch gradient updates.
- **Early Stopping:** Training supports a maximum of 100 epochs, patience of 12 epochs, and restoration of the best model weights based on internal chronological validation loss.

#### Training and Evaluating the Multilayer Perceptron

In [31]:
mlp_validation_predictions = []
mlp_training_summary = []
mlp_training_histories = {}

for fold_number, (train_indices, validation_indices) in enumerate(
    validation_folds,
    start=1
):
    random.seed(random_seed + fold_number)
    np.random.seed(random_seed + fold_number)
    torch.manual_seed(random_seed + fold_number)

    fold_data = prepare_neural_fold(
        train_indices,
        validation_indices
    )

    model = MLPForecaster(
        input_size=sequence_length,
        hidden_size=32,
        output_size=forecast_horizon,
        dropout=0.2
    )

    training_result = train_neural_network(
        model,
        fold_data,
        neural_network_config
    )

    trained_model = training_result["model"]

    X_validation_tensor = torch.tensor(
        fold_data["X_validation"],
        dtype=torch.float32,
        device=device
    )

    trained_model.eval()

    with torch.no_grad():
        scaled_predictions = (
            trained_model(X_validation_tensor)
            .cpu()
            .numpy()
        )

    predictions = fold_data["y_scaler"].inverse_transform(
        scaled_predictions
    )

    actual_values = fold_data["y_validation"]

    assert predictions.shape == actual_values.shape
    assert np.isfinite(predictions).all()

    for sample_number, sample_index in enumerate(validation_indices):
        for horizon in range(1, forecast_horizon + 1):
            mlp_validation_predictions.append({
                "Model": "MLP",
                "Fold": fold_number,
                "Horizon": horizon,
                "Forecast_Origin": train_df.iloc[
                    sample_index
                ]["Forecast_Origin"],
                "Actual": float(
                    actual_values[sample_number, horizon - 1]
                ),
                "Predicted": float(
                    predictions[sample_number, horizon - 1]
                )
            })

    mlp_training_summary.append({
        "Fold": fold_number,
        "Epochs_Trained": training_result["epochs_trained"],
        "Best_Epoch": training_result["best_epoch"],
        "Best_Validation_Loss": training_result["best_validation_loss"]
    })

    mlp_training_histories[fold_number] = training_result["history"]

    print(
        f"Fold {fold_number}: "
        f"Epochs={training_result['epochs_trained']}, "
        f"Best Epoch={training_result['best_epoch']}, "
        f"Best Validation Loss={training_result['best_validation_loss']:.4f}"
    )

mlp_validation_predictions_df = pd.DataFrame(
    mlp_validation_predictions
)

mlp_training_summary_df = pd.DataFrame(
    mlp_training_summary
)

mlp_validation_metrics = []

for (horizon, fold_number), group in (
    mlp_validation_predictions_df.groupby(
        ["Horizon", "Fold"]
    )
):
    mlp_validation_metrics.append({
        "Model": "MLP",
        "Horizon": horizon,
        "Fold": fold_number,
        "MAE": mean_absolute_error(
            group["Actual"],
            group["Predicted"]
        ),
        "RMSE": np.sqrt(
            mean_squared_error(
                group["Actual"],
                group["Predicted"]
            )
        ),
        "R2": r2_score(
            group["Actual"],
            group["Predicted"]
        )
    })

mlp_validation_metrics_df = pd.DataFrame(
    mlp_validation_metrics
)

mlp_summary_df = (
    mlp_validation_metrics_df
    .groupby("Model", as_index=False)
    .agg(
        Mean_MAE=("MAE", "mean"),
        Mean_RMSE=("RMSE", "mean"),
        Mean_R2=("R2", "mean")
    )
)

print("\nMLP Validation Performance:")
display(mlp_summary_df.round(4))

print("\nMLP Training Summary:")
display(mlp_training_summary_df.round(4))

print("\nTotal Forecast Predictions:", len(mlp_validation_predictions_df))
print("Expected Forecast Predictions:", 3 * 14 * forecast_horizon)

assert len(mlp_validation_predictions_df) == 294
assert len(mlp_validation_metrics_df) == 21

print("MLP Evaluation: Completed")

Fold 1: Epochs=53, Best Epoch=41, Best Validation Loss=0.7885
Fold 2: Epochs=51, Best Epoch=39, Best Validation Loss=1.5096
Fold 3: Epochs=49, Best Epoch=37, Best Validation Loss=1.3686

MLP Validation Performance:


,Model,Mean_MAE,Mean_RMSE,Mean_R2
0,MLP,12659.1194,17446.9739,-0.1299



MLP Training Summary:


,Fold,Epochs_Trained,Best_Epoch,Best_Validation_Loss
0,1,53,41,0.7885
1,2,51,39,1.5096
2,3,49,37,1.3686



Total Forecast Predictions: 294
Expected Forecast Predictions: 294
MLP Evaluation: Completed


**Key Observations**
- **MLP Forecasting Performance:** The MLP achieved an average MAE of 12,659.12 and RMSE of 17,446.97, improving upon the Seasonal Naive baseline by approximately 4.06%, although its negative mean R² (-0.1299) indicates limited predictive performance.
- **Training Convergence:** Early stopping selected the best models between epochs 37 and 41, with training ending between epochs 49 and 53 across the three validation folds, demonstrating controlled training without exhausting the 100-epoch limit.

#### Defining the Long Short-Term Memory Architecture

In [32]:
class LSTMForecaster(nn.Module):
    def __init__(
        self,
        input_size=1,
        hidden_size=16,
        output_size=7,
        dropout=0.2
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True
        )

        self.dropout = nn.Dropout(dropout)

        self.output_layer = nn.Linear(
            hidden_size,
            output_size
        )

    def forward(self, x):
        x = x.unsqueeze(-1)

        _, (hidden_state, _) = self.lstm(x)

        final_hidden_state = hidden_state[-1]

        output = self.output_layer(
            self.dropout(final_hidden_state)
        )

        return output

lstm_model = LSTMForecaster(
    input_size=1,
    hidden_size=16,
    output_size=forecast_horizon,
    dropout=0.2
).to(device)

sample_input = torch.tensor(
    neural_X[:2],
    dtype=torch.float32,
    device=device
)

with torch.no_grad():
    sample_output = lstm_model(sample_input)

total_parameters = sum(
    parameter.numel()
    for parameter in lstm_model.parameters()
    if parameter.requires_grad
)

print("LSTM Architecture:")
print(lstm_model)

print("\nInput Shape:", tuple(sample_input.shape))
print("Output Shape:", tuple(sample_output.shape))
print("Trainable Parameters:", total_parameters)
print("Training Device:", device)

assert sample_output.shape == (2, forecast_horizon)

print("\nArchitecture Validation: Passed")

LSTM Architecture:
LSTMForecaster(
  (lstm): LSTM(1, 16, batch_first=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (output_layer): Linear(in_features=16, out_features=7, bias=True)
)

Input Shape: (2, 24)
Output Shape: (2, 7)
Trainable Parameters: 1335
Training Device: cpu

Architecture Validation: Passed


**Key Observations**
- **LSTM Architecture:** The network uses a single LSTM layer with 16 hidden units, followed by 20% dropout regularization and a fully connected output layer for seven-step revenue forecasting.
- **Architecture Validation:** The LSTM contains 1,335 trainable parameters and successfully transforms sequences of 24 historical revenue observations into seven forecast outputs using the CPU.

#### Training and Evaluating the Long Short-Term Memory Model

In [33]:
lstm_validation_predictions = []
lstm_training_summary = []
lstm_training_histories = {}

for fold_number, (train_indices, validation_indices) in enumerate(
    validation_folds,
    start=1
):
    random.seed(random_seed + fold_number)
    np.random.seed(random_seed + fold_number)
    torch.manual_seed(random_seed + fold_number)

    fold_data = prepare_neural_fold(
        train_indices,
        validation_indices
    )

    model = LSTMForecaster(
        input_size=1,
        hidden_size=16,
        output_size=forecast_horizon,
        dropout=0.2
    )

    training_result = train_neural_network(
        model,
        fold_data,
        neural_network_config
    )

    trained_model = training_result["model"]

    X_validation_tensor = torch.tensor(
        fold_data["X_validation"],
        dtype=torch.float32,
        device=device
    )

    trained_model.eval()

    with torch.no_grad():
        scaled_predictions = (
            trained_model(X_validation_tensor)
            .cpu()
            .numpy()
        )

    predictions = fold_data["y_scaler"].inverse_transform(
        scaled_predictions
    )

    actual_values = fold_data["y_validation"]

    assert predictions.shape == actual_values.shape
    assert np.isfinite(predictions).all()

    for sample_number, sample_index in enumerate(validation_indices):
        for horizon in range(1, forecast_horizon + 1):
            lstm_validation_predictions.append({
                "Model": "LSTM",
                "Fold": fold_number,
                "Horizon": horizon,
                "Forecast_Origin": train_df.iloc[
                    sample_index
                ]["Forecast_Origin"],
                "Actual": float(
                    actual_values[sample_number, horizon - 1]
                ),
                "Predicted": float(
                    predictions[sample_number, horizon - 1]
                )
            })

    lstm_training_summary.append({
        "Fold": fold_number,
        "Epochs_Trained": training_result["epochs_trained"],
        "Best_Epoch": training_result["best_epoch"],
        "Best_Validation_Loss": training_result["best_validation_loss"]
    })

    lstm_training_histories[fold_number] = training_result["history"]

    print(
        f"Fold {fold_number}: "
        f"Epochs={training_result['epochs_trained']}, "
        f"Best Epoch={training_result['best_epoch']}, "
        f"Best Validation Loss={training_result['best_validation_loss']:.4f}"
    )

lstm_validation_predictions_df = pd.DataFrame(
    lstm_validation_predictions
)

lstm_training_summary_df = pd.DataFrame(
    lstm_training_summary
)

lstm_validation_metrics = []

for (horizon, fold_number), group in (
    lstm_validation_predictions_df.groupby(
        ["Horizon", "Fold"]
    )
):
    lstm_validation_metrics.append({
        "Model": "LSTM",
        "Horizon": horizon,
        "Fold": fold_number,
        "MAE": mean_absolute_error(
            group["Actual"],
            group["Predicted"]
        ),
        "RMSE": np.sqrt(
            mean_squared_error(
                group["Actual"],
                group["Predicted"]
            )
        ),
        "R2": r2_score(
            group["Actual"],
            group["Predicted"]
        )
    })

lstm_validation_metrics_df = pd.DataFrame(
    lstm_validation_metrics
)

lstm_summary_df = (
    lstm_validation_metrics_df
    .groupby("Model", as_index=False)
    .agg(
        Mean_MAE=("MAE", "mean"),
        Mean_RMSE=("RMSE", "mean"),
        Mean_R2=("R2", "mean")
    )
)

print("\nLSTM Validation Performance:")
display(lstm_summary_df.round(4))

print("\nLSTM Training Summary:")
display(lstm_training_summary_df.round(4))

print("\nTotal Forecast Predictions:", len(lstm_validation_predictions_df))
print("Expected Forecast Predictions:", 3 * 14 * forecast_horizon)

assert len(lstm_validation_predictions_df) == 294
assert len(lstm_validation_metrics_df) == 21

print("LSTM Evaluation: Completed")

Fold 1: Epochs=20, Best Epoch=8, Best Validation Loss=0.9749
Fold 2: Epochs=51, Best Epoch=39, Best Validation Loss=1.7392
Fold 3: Epochs=74, Best Epoch=62, Best Validation Loss=1.4571

LSTM Validation Performance:


,Model,Mean_MAE,Mean_RMSE,Mean_R2
0,LSTM,13379.5098,18489.6337,-0.2081



LSTM Training Summary:


,Fold,Epochs_Trained,Best_Epoch,Best_Validation_Loss
0,1,20,8,0.9749
1,2,51,39,1.7392
2,3,74,62,1.4571



Total Forecast Predictions: 294
Expected Forecast Predictions: 294
LSTM Evaluation: Completed


**Key Observations**
- **LSTM Forecasting Performance:** The LSTM achieved an average MAE of 13,379.51 and RMSE of 18,489.63, performing worse than the MLP (MAE: 12,659.12) and the Seasonal Naive baseline (MAE: 13,194.90).
- **Training Behavior:** Early stopping selected the best epochs at 8, 39, and 62 across the three folds, indicating varying convergence behavior. The negative mean R² (-0.2081) suggests limited predictive effectiveness with the available historical sequences.

#### Defining the Gated Recurrent Unit Architecture

In [34]:
class GRUForecaster(nn.Module):
    def __init__(
        self,
        input_size=1,
        hidden_size=16,
        output_size=7,
        dropout=0.2
    ):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True
        )

        self.dropout = nn.Dropout(dropout)

        self.output_layer = nn.Linear(
            hidden_size,
            output_size
        )

    def forward(self, x):
        x = x.unsqueeze(-1)

        _, hidden_state = self.gru(x)

        final_hidden_state = hidden_state[-1]

        output = self.output_layer(
            self.dropout(final_hidden_state)
        )

        return output

gru_model = GRUForecaster(
    input_size=1,
    hidden_size=16,
    output_size=forecast_horizon,
    dropout=0.2
).to(device)

sample_input = torch.tensor(
    neural_X[:2],
    dtype=torch.float32,
    device=device
)

with torch.no_grad():
    sample_output = gru_model(sample_input)

total_parameters = sum(
    parameter.numel()
    for parameter in gru_model.parameters()
    if parameter.requires_grad
)

print("GRU Architecture:")
print(gru_model)

print("\nInput Shape:", tuple(sample_input.shape))
print("Output Shape:", tuple(sample_output.shape))
print("Trainable Parameters:", total_parameters)
print("Training Device:", device)

assert sample_output.shape == (2, forecast_horizon)

print("\nArchitecture Validation: Passed")

GRU Architecture:
GRUForecaster(
  (gru): GRU(1, 16, batch_first=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (output_layer): Linear(in_features=16, out_features=7, bias=True)
)

Input Shape: (2, 24)
Output Shape: (2, 7)
Trainable Parameters: 1031
Training Device: cpu

Architecture Validation: Passed


**Key Observations**
- **GRU Architecture:** The network uses a single GRU layer with 16 hidden units, followed by 20% dropout regularization and a fully connected layer producing seven revenue forecasts.
- **Architecture Validation:** The GRU contains 1,031 trainable parameters, fewer than both the MLP and LSTM, and successfully transforms 24 historical revenue observations into seven forecast outputs using the CPU.

#### Training and Evaluating the Gated Recurrent Unit Model

In [35]:
gru_validation_predictions = []
gru_training_summary = []
gru_training_histories = {}

for fold_number, (train_indices, validation_indices) in enumerate(
    validation_folds,
    start=1
):
    random.seed(random_seed + fold_number)
    np.random.seed(random_seed + fold_number)
    torch.manual_seed(random_seed + fold_number)

    fold_data = prepare_neural_fold(
        train_indices,
        validation_indices
    )

    model = GRUForecaster(
        input_size=1,
        hidden_size=16,
        output_size=forecast_horizon,
        dropout=0.2
    )

    training_result = train_neural_network(
        model,
        fold_data,
        neural_network_config
    )

    trained_model = training_result["model"]

    X_validation_tensor = torch.tensor(
        fold_data["X_validation"],
        dtype=torch.float32,
        device=device
    )

    trained_model.eval()

    with torch.no_grad():
        scaled_predictions = (
            trained_model(X_validation_tensor)
            .cpu()
            .numpy()
        )

    predictions = fold_data["y_scaler"].inverse_transform(
        scaled_predictions
    )

    actual_values = fold_data["y_validation"]

    assert predictions.shape == actual_values.shape
    assert np.isfinite(predictions).all()

    for sample_number, sample_index in enumerate(validation_indices):
        for horizon in range(1, forecast_horizon + 1):
            gru_validation_predictions.append({
                "Model": "GRU",
                "Fold": fold_number,
                "Horizon": horizon,
                "Forecast_Origin": train_df.iloc[
                    sample_index
                ]["Forecast_Origin"],
                "Actual": float(
                    actual_values[sample_number, horizon - 1]
                ),
                "Predicted": float(
                    predictions[sample_number, horizon - 1]
                )
            })

    gru_training_summary.append({
        "Fold": fold_number,
        "Epochs_Trained": training_result["epochs_trained"],
        "Best_Epoch": training_result["best_epoch"],
        "Best_Validation_Loss": training_result["best_validation_loss"]
    })

    gru_training_histories[fold_number] = training_result["history"]

    print(
        f"Fold {fold_number}: "
        f"Epochs={training_result['epochs_trained']}, "
        f"Best Epoch={training_result['best_epoch']}, "
        f"Best Validation Loss={training_result['best_validation_loss']:.4f}"
    )

gru_validation_predictions_df = pd.DataFrame(
    gru_validation_predictions
)

gru_training_summary_df = pd.DataFrame(
    gru_training_summary
)

gru_validation_metrics = []

for (horizon, fold_number), group in (
    gru_validation_predictions_df.groupby(
        ["Horizon", "Fold"]
    )
):
    gru_validation_metrics.append({
        "Model": "GRU",
        "Horizon": horizon,
        "Fold": fold_number,
        "MAE": mean_absolute_error(
            group["Actual"],
            group["Predicted"]
        ),
        "RMSE": np.sqrt(
            mean_squared_error(
                group["Actual"],
                group["Predicted"]
            )
        ),
        "R2": r2_score(
            group["Actual"],
            group["Predicted"]
        )
    })

gru_validation_metrics_df = pd.DataFrame(
    gru_validation_metrics
)

gru_summary_df = (
    gru_validation_metrics_df
    .groupby("Model", as_index=False)
    .agg(
        Mean_MAE=("MAE", "mean"),
        Mean_RMSE=("RMSE", "mean"),
        Mean_R2=("R2", "mean")
    )
)

print("\nGRU Validation Performance:")
display(gru_summary_df.round(4))

print("\nGRU Training Summary:")
display(gru_training_summary_df.round(4))

print("\nTotal Forecast Predictions:", len(gru_validation_predictions_df))
print("Expected Forecast Predictions:", 3 * 14 * forecast_horizon)

assert len(gru_validation_predictions_df) == 294
assert len(gru_validation_metrics_df) == 21

print("GRU Evaluation: Completed")

Fold 1: Epochs=35, Best Epoch=23, Best Validation Loss=0.9900
Fold 2: Epochs=96, Best Epoch=84, Best Validation Loss=1.4785
Fold 3: Epochs=60, Best Epoch=48, Best Validation Loss=1.4731

GRU Validation Performance:


,Model,Mean_MAE,Mean_RMSE,Mean_R2
0,GRU,13487.3844,18524.6401,-0.2181



GRU Training Summary:


,Fold,Epochs_Trained,Best_Epoch,Best_Validation_Loss
0,1,35,23,0.9900
1,2,96,84,1.4785
2,3,60,48,1.4731



Total Forecast Predictions: 294
Expected Forecast Predictions: 294
GRU Evaluation: Completed


**Key Observations**
- **GRU Forecasting Performance:** The GRU achieved an average MAE of 13,487.38 and RMSE of 18,524.64, performing slightly worse than the LSTM (MAE: 13,379.51) and MLP (MAE: 12,659.12), with a negative mean R² of -0.2181.
- **Training Behavior:** Early stopping selected the best epochs at 23, 84, and 48 across the three folds, indicating considerable variation in convergence while successfully completing all 294 forecast predictions.

#### Defining the One-Dimensional Convolutional Neural Network Architecture

In [36]:
class CNN1DForecaster(nn.Module):
    def __init__(
        self,
        sequence_length=24,
        output_size=7,
        dropout=0.2
    ):
        super().__init__()

        self.feature_extractor = nn.Sequential(
            nn.Conv1d(
                in_channels=1,
                out_channels=16,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.Conv1d(
                in_channels=16,
                out_channels=8,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(6)
        )

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(8 * 6, 16),
            nn.ReLU(),
            nn.Linear(16, output_size)
        )

    def forward(self, x):
        x = x.unsqueeze(1)

        features = self.feature_extractor(x)

        output = self.regressor(features)

        return output

cnn_model = CNN1DForecaster(
    sequence_length=sequence_length,
    output_size=forecast_horizon,
    dropout=0.2
).to(device)

sample_input = torch.tensor(
    neural_X[:2],
    dtype=torch.float32,
    device=device
)

with torch.no_grad():
    sample_output = cnn_model(sample_input)

total_parameters = sum(
    parameter.numel()
    for parameter in cnn_model.parameters()
    if parameter.requires_grad
)

print("1D CNN Architecture:")
print(cnn_model)

print("\nInput Shape:", tuple(sample_input.shape))
print("Output Shape:", tuple(sample_output.shape))
print("Trainable Parameters:", total_parameters)
print("Training Device:", device)

assert sample_output.shape == (2, forecast_horizon)

print("\nArchitecture Validation: Passed")

1D CNN Architecture:
CNN1DForecaster(
  (feature_extractor): Sequential(
    (0): Conv1d(1, 16, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): ReLU()
    (2): Conv1d(16, 8, kernel_size=(3,), stride=(1,), padding=(1,))
    (3): ReLU()
    (4): AdaptiveAvgPool1d(output_size=6)
  )
  (regressor): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Dropout(p=0.2, inplace=False)
    (2): Linear(in_features=48, out_features=16, bias=True)
    (3): ReLU()
    (4): Linear(in_features=16, out_features=7, bias=True)
  )
)

Input Shape: (2, 24)
Output Shape: (2, 7)
Trainable Parameters: 1359
Training Device: cpu

Architecture Validation: Passed


**Key Observations**
- **CNN Architecture:** The network uses two convolutional layers with 16 and 8 filters, followed by adaptive average pooling, dropout regularization, and fully connected layers for seven-step revenue forecasting.
- **Architecture Validation:** The CNN contains 1,359 trainable parameters and successfully transforms 24 historical revenue observations into seven forecast outputs using the CPU.

#### Training and Evaluating the One-Dimensional Convolutional Neural Network

In [37]:
cnn_validation_predictions = []
cnn_training_summary = []
cnn_training_histories = {}

for fold_number, (train_indices, validation_indices) in enumerate(
    validation_folds,
    start=1
):
    random.seed(random_seed + fold_number)
    np.random.seed(random_seed + fold_number)
    torch.manual_seed(random_seed + fold_number)

    fold_data = prepare_neural_fold(
        train_indices,
        validation_indices
    )

    model = CNN1DForecaster(
        sequence_length=sequence_length,
        output_size=forecast_horizon,
        dropout=0.2
    )

    training_result = train_neural_network(
        model,
        fold_data,
        neural_network_config
    )

    trained_model = training_result["model"]

    X_validation_tensor = torch.tensor(
        fold_data["X_validation"],
        dtype=torch.float32,
        device=device
    )

    trained_model.eval()

    with torch.no_grad():
        scaled_predictions = (
            trained_model(X_validation_tensor)
            .cpu()
            .numpy()
        )

    predictions = fold_data["y_scaler"].inverse_transform(
        scaled_predictions
    )

    actual_values = fold_data["y_validation"]

    assert predictions.shape == actual_values.shape
    assert np.isfinite(predictions).all()

    for sample_number, sample_index in enumerate(validation_indices):
        for horizon in range(1, forecast_horizon + 1):
            cnn_validation_predictions.append({
                "Model": "CNN_1D",
                "Fold": fold_number,
                "Horizon": horizon,
                "Forecast_Origin": train_df.iloc[
                    sample_index
                ]["Forecast_Origin"],
                "Actual": float(
                    actual_values[sample_number, horizon - 1]
                ),
                "Predicted": float(
                    predictions[sample_number, horizon - 1]
                )
            })

    cnn_training_summary.append({
        "Fold": fold_number,
        "Epochs_Trained": training_result["epochs_trained"],
        "Best_Epoch": training_result["best_epoch"],
        "Best_Validation_Loss": training_result["best_validation_loss"]
    })

    cnn_training_histories[fold_number] = training_result["history"]

    print(
        f"Fold {fold_number}: "
        f"Epochs={training_result['epochs_trained']}, "
        f"Best Epoch={training_result['best_epoch']}, "
        f"Best Validation Loss={training_result['best_validation_loss']:.4f}"
    )

cnn_validation_predictions_df = pd.DataFrame(
    cnn_validation_predictions
)

cnn_training_summary_df = pd.DataFrame(
    cnn_training_summary
)

cnn_validation_metrics = []

for (horizon, fold_number), group in (
    cnn_validation_predictions_df.groupby(
        ["Horizon", "Fold"]
    )
):
    cnn_validation_metrics.append({
        "Model": "CNN_1D",
        "Horizon": horizon,
        "Fold": fold_number,
        "MAE": mean_absolute_error(
            group["Actual"],
            group["Predicted"]
        ),
        "RMSE": np.sqrt(
            mean_squared_error(
                group["Actual"],
                group["Predicted"]
            )
        ),
        "R2": r2_score(
            group["Actual"],
            group["Predicted"]
        )
    })

cnn_validation_metrics_df = pd.DataFrame(
    cnn_validation_metrics
)

cnn_summary_df = (
    cnn_validation_metrics_df
    .groupby("Model", as_index=False)
    .agg(
        Mean_MAE=("MAE", "mean"),
        Mean_RMSE=("RMSE", "mean"),
        Mean_R2=("R2", "mean")
    )
)

print("\n1D CNN Validation Performance:")
display(cnn_summary_df.round(4))

print("\n1D CNN Training Summary:")
display(cnn_training_summary_df.round(4))

print("\nTotal Forecast Predictions:", len(cnn_validation_predictions_df))
print("Expected Forecast Predictions:", 3 * 14 * forecast_horizon)

assert len(cnn_validation_predictions_df) == 294
assert len(cnn_validation_metrics_df) == 21

print("1D CNN Evaluation: Completed")

Fold 1: Epochs=56, Best Epoch=44, Best Validation Loss=0.9011
Fold 2: Epochs=13, Best Epoch=1, Best Validation Loss=1.7683
Fold 3: Epochs=13, Best Epoch=1, Best Validation Loss=1.8789

1D CNN Validation Performance:


,Model,Mean_MAE,Mean_RMSE,Mean_R2
0,CNN_1D,12951.6042,18122.6343,-0.1606



1D CNN Training Summary:


,Fold,Epochs_Trained,Best_Epoch,Best_Validation_Loss
0,1,56,44,0.9011
1,2,13,1,1.7683
2,3,13,1,1.8789



Total Forecast Predictions: 294
Expected Forecast Predictions: 294
1D CNN Evaluation: Completed


**Key Observations**
- **CNN Forecasting Performance:** The 1D CNN achieved an average MAE of 12,951.60 and RMSE of 18,122.63, outperforming the LSTM and GRU but remaining behind the MLP (MAE: 12,659.12). Its mean R² of -0.1606 indicates limited predictive effectiveness.
- **Training Behavior:** Early stopping selected epoch 44 in the first fold but epoch 1 in the remaining two folds, suggesting unstable generalization across validation periods despite completing all 294 forecast predictions.

#### Comparing Neural Network Forecasting Performance

In [38]:
neural_validation_metrics_df = pd.concat(
    [
        mlp_validation_metrics_df,
        lstm_validation_metrics_df,
        gru_validation_metrics_df,
        cnn_validation_metrics_df
    ],
    ignore_index=True
)

neural_validation_predictions_df = pd.concat(
    [
        mlp_validation_predictions_df,
        lstm_validation_predictions_df,
        gru_validation_predictions_df,
        cnn_validation_predictions_df
    ],
    ignore_index=True
)

neural_model_summary_df = (
    neural_validation_metrics_df
    .groupby("Model", as_index=False)
    .agg(
        Mean_MAE=("MAE", "mean"),
        Mean_RMSE=("RMSE", "mean"),
        Mean_R2=("R2", "mean")
    )
)

neural_model_summary_df["Improvement_vs_Seasonal_Naive_%"] = (
    (
        seasonal_baseline_mae
        - neural_model_summary_df["Mean_MAE"]
    )
    / seasonal_baseline_mae
    * 100
)

neural_model_summary_df = (
    neural_model_summary_df
    .sort_values("Mean_MAE")
    .reset_index(drop=True)
)

neural_model_summary_df.insert(
    0,
    "Rank",
    np.arange(1, len(neural_model_summary_df) + 1)
)

print("Neural Network Model Comparison:")
display(neural_model_summary_df.round(4))

print("\nTotal Neural Network Models:", neural_model_summary_df["Model"].nunique())
print("Total Validation Predictions:", len(neural_validation_predictions_df))
print("Total Fold-Horizon Metric Rows:", len(neural_validation_metrics_df))

assert neural_model_summary_df["Model"].nunique() == 4
assert len(neural_validation_predictions_df) == 1176
assert len(neural_validation_metrics_df) == 84

print("\nNeural Network Comparison: Completed")

Neural Network Model Comparison:


,Rank,Model,Mean_MAE,Mean_RMSE,Mean_R2,Improvement_vs_Seasonal_Naive_%
0,1,MLP,12659.1194,17446.9739,-0.1299,4.0605
1,2,CNN_1D,12951.6042,18122.6343,-0.1606,1.8439
2,3,LSTM,13379.5098,18489.6337,-0.2081,-1.3991
3,4,GRU,13487.3844,18524.6401,-0.2181,-2.2166



Total Neural Network Models: 4
Total Validation Predictions: 1176
Total Fold-Horizon Metric Rows: 84

Neural Network Comparison: Completed


**Key Observations**
- **Neural Network Performance:** The MLP achieved the lowest average MAE (12,659.12), followed by the 1D CNN (12,951.60), while the LSTM and GRU produced higher forecasting errors.
- **Baseline Comparison:** The MLP and 1D CNN improved upon the Seasonal Naive baseline by 4.06% and 1.84%, respectively, whereas the LSTM and GRU underperformed the baseline. All four models recorded negative mean R² values.

#### Comparing Forecasting Models Across All Approaches

In [39]:
all_model_comparison_df = pd.concat(
    [
        combined_model_comparison_df[
            ["Model", "Mean_MAE", "Mean_RMSE", "Mean_R2"]
        ],
        neural_model_summary_df[
            ["Model", "Mean_MAE", "Mean_RMSE", "Mean_R2"]
        ]
    ],
    ignore_index=True
)

all_model_comparison_df["Improvement_vs_Seasonal_Naive_%"] = (
    (
        seasonal_baseline_mae
        - all_model_comparison_df["Mean_MAE"]
    )
    / seasonal_baseline_mae
    * 100
)

all_model_comparison_df = (
    all_model_comparison_df
    .sort_values("Mean_MAE")
    .reset_index(drop=True)
)

all_model_comparison_df.insert(
    0,
    "Rank",
    np.arange(1, len(all_model_comparison_df) + 1)
)

print("Overall Forecasting Model Comparison:")
display(all_model_comparison_df.round(4))

print("\nTotal Models Evaluated:", len(all_model_comparison_df))
print(
    "Models Outperforming Seasonal Naive:",
    (
        all_model_comparison_df["Mean_MAE"]
        < seasonal_baseline_mae
    ).sum()
)

print("\nTop Five Forecasting Models:")
display(all_model_comparison_df.head(5).round(4))

assert len(all_model_comparison_df) == 21
assert all_model_comparison_df["Model"].is_unique

print("\nOverall Model Comparison: Completed")

Overall Forecasting Model Comparison:


,Rank,Model,Mean_MAE,Mean_RMSE,Mean_R2,Improvement_vs_Seasonal_Naive_%
0,1,AutoReg,11812.5487,16419.5086,0.0166,10.4764
1,2,SARIMA,11980.8481,16715.9763,-0.1623,9.2010
2,3,Random_Forest,12008.9388,16806.5875,-0.0111,8.9881
3,4,ElasticNet,12078.3441,16616.4295,0.0322,8.4621
4,5,ETS,12189.1787,16657.1399,-0.1456,7.6221
5,6,Extra_Trees,12269.2499,16949.4003,-0.0539,7.0152
6,7,Hist_Gradient_Boosting,12460.0799,17179.9529,-0.0751,5.5690
7,8,LightGBM,12553.1895,17298.2834,-0.1093,4.8634
8,9,MLP,12659.1194,17446.9739,-0.1299,4.0605
9,10,Ridge,12669.5148,17215.4998,-0.0663,3.9818



Total Models Evaluated: 21
Models Outperforming Seasonal Naive: 15

Top Five Forecasting Models:


,Rank,Model,Mean_MAE,Mean_RMSE,Mean_R2,Improvement_vs_Seasonal_Naive_%
0,1,AutoReg,11812.5487,16419.5086,0.0166,10.4764
1,2,SARIMA,11980.8481,16715.9763,-0.1623,9.2010
2,3,Random_Forest,12008.9388,16806.5875,-0.0111,8.9881
3,4,ElasticNet,12078.3441,16616.4295,0.0322,8.4621
4,5,ETS,12189.1787,16657.1399,-0.1456,7.6221



Overall Model Comparison: Completed


**Key Observations**
- **Leading Forecasting Models:** AutoReg achieved the lowest average MAE (11,812.55), followed by SARIMA (11,980.85), Random Forest (12,008.94), and ElasticNet (12,078.34), demonstrating competitive performance across statistical and machine learning approaches.
- **Overall Model Performance:** Of the 21 evaluated models, 15 outperformed the Seasonal Naive baseline, with AutoReg delivering the largest MAE improvement (10.48%). Neural networks generally ranked below the leading statistical and classical machine learning models.

#### Selecting Forecasting Models for Hyperparameter Optimization

In [40]:
tuning_candidates = [
    "AutoReg",
    "SARIMA",
    "Random_Forest",
    "ElasticNet",
    "Extra_Trees"
]

tuning_candidates_df = (
    all_model_comparison_df[
        all_model_comparison_df["Model"].isin(tuning_candidates)
    ]
    .copy()
    .sort_values("Mean_MAE")
    .reset_index(drop=True)
)

tuning_candidates_df["Model_Family"] = (
    tuning_candidates_df["Model"].map({
        "AutoReg": "Statistical",
        "SARIMA": "Statistical",
        "Random_Forest": "Tree-Based ML",
        "ElasticNet": "Linear ML",
        "Extra_Trees": "Tree-Based ML"
    })
)

tuning_candidates_df["Tuning_Priority"] = (
    np.arange(1, len(tuning_candidates_df) + 1)
)

tuning_candidates_df = tuning_candidates_df[
    [
        "Tuning_Priority",
        "Model",
        "Model_Family",
        "Mean_MAE",
        "Mean_RMSE",
        "Mean_R2",
        "Improvement_vs_Seasonal_Naive_%"
    ]
]

print("Selected Hyperparameter Tuning Candidates:")
display(tuning_candidates_df.round(4))

print("\nTotal Models Selected:", len(tuning_candidates_df))
print("Model Families Represented:", tuning_candidates_df["Model_Family"].nunique())

assert len(tuning_candidates_df) == 5
assert tuning_candidates_df["Model"].is_unique

print("\nHyperparameter Tuning Candidate Selection: Completed")

Selected Hyperparameter Tuning Candidates:


,Tuning_Priority,Model,Model_Family,Mean_MAE,Mean_RMSE,Mean_R2,Improvement_vs_Seasonal_Naive_%
0,1,AutoReg,Statistical,11812.5487,16419.5086,0.0166,10.4764
1,2,SARIMA,Statistical,11980.8481,16715.9763,-0.1623,9.2010
2,3,Random_Forest,Tree-Based ML,12008.9388,16806.5875,-0.0111,8.9881
3,4,ElasticNet,Linear ML,12078.3441,16616.4295,0.0322,8.4621
4,5,Extra_Trees,Tree-Based ML,12269.2499,16949.4003,-0.0539,7.0152



Total Models Selected: 5
Model Families Represented: 3

Hyperparameter Tuning Candidate Selection: Completed


**Key Observations**
- **Tuning Candidates:** Five models were selected for hyperparameter optimization, with AutoReg (MAE: 11,812.55), SARIMA (11,980.85), and Random Forest (12,008.94) leading the shortlist.
- **Model Diversity:** The selected candidates represent statistical, tree-based, and regularized linear forecasting approaches, allowing optimization across different modeling techniques while maintaining computational efficiency.

#### Configuring Hyperparameter Search Spaces

In [41]:
hyperparameter_search_spaces = {
    "AutoReg": {
        "lags": [
            [1, 2, 3, 6],
            [1, 2, 3, 6, 12],
            [1, 2, 3, 6, 12, 18]
        ]
    },
    "SARIMA": {
        "order": [
            (1, 0, 0),
            (1, 0, 1),
            (2, 0, 0)
        ],
        "seasonal_order": [
            (1, 0, 0, 6),
            (1, 0, 1, 6)
        ]
    },
    "Random_Forest": {
        "n_estimators": [100, 200],
        "max_depth": [5, 10, None],
        "min_samples_leaf": [2, 5]
    },
    "ElasticNet": {
        "alpha": [0.1, 1.0, 10.0],
        "l1_ratio": [0.2, 0.5, 0.8]
    },
    "Extra_Trees": {
        "n_estimators": [100, 200],
        "max_depth": [5, 10, None],
        "min_samples_leaf": [2, 5]
    }
}

hyperparameter_search_summary = []

for model_name, search_space in hyperparameter_search_spaces.items():
    configurations = list(ParameterGrid(search_space))

    hyperparameter_search_summary.append({
        "Model": model_name,
        "Configurations": len(configurations),
        "Validation_Folds": len(validation_folds),
        "Total_Configuration_Fold_Evaluations": (
            len(configurations) * len(validation_folds)
        )
    })

hyperparameter_search_summary_df = pd.DataFrame(
    hyperparameter_search_summary
)

print("Hyperparameter Search Configuration:")
display(hyperparameter_search_summary_df)

print(
    "\nTotal Configurations:",
    hyperparameter_search_summary_df["Configurations"].sum()
)

print(
    "Total Configuration-Fold Evaluations:",
    hyperparameter_search_summary_df[
        "Total_Configuration_Fold_Evaluations"
    ].sum()
)

assert set(hyperparameter_search_spaces) == set(tuning_candidates)
assert hyperparameter_search_summary_df["Configurations"].sum() == 42

print("\nHyperparameter Search Configuration: Completed")

Hyperparameter Search Configuration:


,Model,Configurations,Validation_Folds,Total_Configuration_Fold_Evaluations
0,AutoReg,3,3,9
1,SARIMA,6,3,18
2,Random_Forest,12,3,36
3,ElasticNet,9,3,27
4,Extra_Trees,12,3,36



Total Configurations: 42
Total Configuration-Fold Evaluations: 126

Hyperparameter Search Configuration: Completed


**Key Observations**
- **Hyperparameter Search Scope:** Five forecasting models have been assigned 42 hyperparameter configurations, covering autoregressive lags, seasonal ARIMA orders, tree complexity, and ElasticNet regularization.
- **Validation Strategy:** The search includes 126 configuration-fold evaluations across three expanding-window folds, providing a structured comparison while limiting computational overhead.

#### Tuning the Autoregressive Forecasting Model

In [43]:
autoreg_tuning_results = []
autoreg_tuning_predictions = []

autoreg_configurations = list(
    ParameterGrid(hyperparameter_search_spaces["AutoReg"])
)

for config_number, parameters in enumerate(
    autoreg_configurations,
    start=1
):
    lags = parameters["lags"]

    for fold_number, (_, validation_indices) in enumerate(
        validation_folds,
        start=1
    ):
        fold_predictions = []

        for sample_index in validation_indices:
            origin_position = int(origin_positions[sample_index])

            history = statistical_series.iloc[
                :origin_position + 1
            ]

            actual_values = statistical_series.iloc[
                origin_position + 1:
                origin_position + forecast_horizon + 1
            ].to_numpy(dtype=float)

            with warnings.catch_warnings():
                warnings.simplefilter("ignore")

                fitted_model = AutoReg(
                    history,
                    lags=lags,
                    trend="c"
                ).fit()

                predictions = np.asarray(
                    fitted_model.forecast(steps=forecast_horizon),
                    dtype=float
                )

            assert len(predictions) == forecast_horizon
            assert np.isfinite(predictions).all()

            for horizon in range(1, forecast_horizon + 1):
                record = {
                    "Model": "AutoReg",
                    "Configuration": config_number,
                    "Fold": fold_number,
                    "Horizon": horizon,
                    "Forecast_Origin": train_df.iloc[
                        sample_index
                    ]["Forecast_Origin"],
                    "Actual": float(actual_values[horizon - 1]),
                    "Predicted": float(predictions[horizon - 1])
                }

                fold_predictions.append(record)
                autoreg_tuning_predictions.append(record)

        fold_predictions_df = pd.DataFrame(fold_predictions)

        for horizon, group in fold_predictions_df.groupby("Horizon"):
            autoreg_tuning_results.append({
                "Model": "AutoReg",
                "Configuration": config_number,
                "Lags": str(lags),
                "Fold": fold_number,
                "Horizon": horizon,
                "MAE": mean_absolute_error(
                    group["Actual"],
                    group["Predicted"]
                ),
                "RMSE": np.sqrt(
                    mean_squared_error(
                        group["Actual"],
                        group["Predicted"]
                    )
                ),
                "R2": r2_score(
                    group["Actual"],
                    group["Predicted"]
                )
            })

    print(
        f"Configuration {config_number}/{len(autoreg_configurations)} "
        f"completed | Lags: {lags}"
    )

autoreg_tuning_results_df = pd.DataFrame(
    autoreg_tuning_results
)

autoreg_tuning_predictions_df = pd.DataFrame(
    autoreg_tuning_predictions
)

autoreg_tuning_summary_df = (
    autoreg_tuning_results_df
    .groupby(["Configuration", "Lags"], as_index=False)
    .agg(
        Mean_MAE=("MAE", "mean"),
        Mean_RMSE=("RMSE", "mean"),
        Mean_R2=("R2", "mean")
    )
    .sort_values("Mean_MAE")
    .reset_index(drop=True)
)

best_autoreg_configuration = int(
    autoreg_tuning_summary_df.iloc[0]["Configuration"]
)

best_autoreg_parameters = autoreg_configurations[
    best_autoreg_configuration - 1
]

print("\nAutoReg Hyperparameter Tuning Results:")
display(autoreg_tuning_summary_df.round(4))

print("\nBest AutoReg Parameters:", best_autoreg_parameters)
print(
    "Best Validation MAE:",
    round(autoreg_tuning_summary_df.iloc[0]["Mean_MAE"], 4)
)

print("\nTotal Forecast Predictions:", len(autoreg_tuning_predictions_df))
print("Total Fold-Horizon Metric Rows:", len(autoreg_tuning_results_df))

assert len(autoreg_tuning_predictions_df) == 882
assert len(autoreg_tuning_results_df) == 63

print("\nAutoReg Hyperparameter Tuning: Completed")

Configuration 1/3 completed | Lags: [1, 2, 3, 6]
Configuration 2/3 completed | Lags: [1, 2, 3, 6, 12]
Configuration 3/3 completed | Lags: [1, 2, 3, 6, 12, 18]

AutoReg Hyperparameter Tuning Results:


,Configuration,Lags,Mean_MAE,Mean_RMSE,Mean_R2
0,2,"[1, 2, 3, 6, 12]",11812.5487,16419.5086,0.0166
1,3,"[1, 2, 3, 6, 12, 18]",11944.3236,16630.0131,-0.0156
2,1,"[1, 2, 3, 6]",12137.6950,16570.9749,-0.0009



Best AutoReg Parameters: {'lags': [1, 2, 3, 6, 12]}
Best Validation MAE: 11812.5487

Total Forecast Predictions: 882
Total Fold-Horizon Metric Rows: 63

AutoReg Hyperparameter Tuning: Completed


**Key Observations**
- **Optimal Lag Configuration:** AutoReg achieved its lowest average MAE (11,812.55) using lags [1, 2, 3, 6, 12], confirming the original configuration as the best among the three evaluated alternatives.
- **Tuning Outcome:** Adding lag 18 increased MAE to 11,944.32, while removing lag 12 increased it to 12,137.70. Neither alternative improved forecasting accuracy, so the original configuration was retained.

#### Tuning the Seasonal Autoregressive Integrated Moving Average Model

In [44]:
sarima_tuning_results = []
sarima_tuning_predictions = []

sarima_configurations = list(
    ParameterGrid(hyperparameter_search_spaces["SARIMA"])
)

for config_number, parameters in enumerate(
    sarima_configurations,
    start=1
):
    order = parameters["order"]
    seasonal_order = parameters["seasonal_order"]

    for fold_number, (_, validation_indices) in enumerate(
        validation_folds,
        start=1
    ):
        fold_predictions = []

        for sample_index in validation_indices:
            origin_position = int(origin_positions[sample_index])

            history = statistical_series.iloc[
                :origin_position + 1
            ]

            actual_values = statistical_series.iloc[
                origin_position + 1:
                origin_position + forecast_horizon + 1
            ].to_numpy(dtype=float)

            with warnings.catch_warnings():
                warnings.simplefilter("ignore")

                fitted_model = SARIMAX(
                    history,
                    order=order,
                    seasonal_order=seasonal_order,
                    trend="c",
                    enforce_stationarity=False,
                    enforce_invertibility=False
                ).fit(
                    disp=False,
                    maxiter=100
                )

                predictions = np.asarray(
                    fitted_model.forecast(steps=forecast_horizon),
                    dtype=float
                )

            assert len(predictions) == forecast_horizon
            assert np.isfinite(predictions).all()

            for horizon in range(1, forecast_horizon + 1):
                record = {
                    "Model": "SARIMA",
                    "Configuration": config_number,
                    "Fold": fold_number,
                    "Horizon": horizon,
                    "Forecast_Origin": train_df.iloc[
                        sample_index
                    ]["Forecast_Origin"],
                    "Actual": float(actual_values[horizon - 1]),
                    "Predicted": float(predictions[horizon - 1])
                }

                fold_predictions.append(record)
                sarima_tuning_predictions.append(record)

        fold_predictions_df = pd.DataFrame(fold_predictions)

        for horizon, group in fold_predictions_df.groupby("Horizon"):
            sarima_tuning_results.append({
                "Model": "SARIMA",
                "Configuration": config_number,
                "Order": str(order),
                "Seasonal_Order": str(seasonal_order),
                "Fold": fold_number,
                "Horizon": horizon,
                "MAE": mean_absolute_error(
                    group["Actual"],
                    group["Predicted"]
                ),
                "RMSE": np.sqrt(
                    mean_squared_error(
                        group["Actual"],
                        group["Predicted"]
                    )
                ),
                "R2": r2_score(
                    group["Actual"],
                    group["Predicted"]
                )
            })

    print(
        f"Configuration {config_number}/{len(sarima_configurations)} "
        f"completed | Order: {order} | Seasonal Order: {seasonal_order}"
    )

sarima_tuning_results_df = pd.DataFrame(
    sarima_tuning_results
)

sarima_tuning_predictions_df = pd.DataFrame(
    sarima_tuning_predictions
)

sarima_tuning_summary_df = (
    sarima_tuning_results_df
    .groupby(
        ["Configuration", "Order", "Seasonal_Order"],
        as_index=False
    )
    .agg(
        Mean_MAE=("MAE", "mean"),
        Mean_RMSE=("RMSE", "mean"),
        Mean_R2=("R2", "mean")
    )
    .sort_values("Mean_MAE")
    .reset_index(drop=True)
)

best_sarima_configuration = int(
    sarima_tuning_summary_df.iloc[0]["Configuration"]
)

best_sarima_parameters = sarima_configurations[
    best_sarima_configuration - 1
]

print("\nSARIMA Hyperparameter Tuning Results:")
display(sarima_tuning_summary_df.round(4))

print("\nBest SARIMA Parameters:", best_sarima_parameters)
print(
    "Best Validation MAE:",
    round(sarima_tuning_summary_df.iloc[0]["Mean_MAE"], 4)
)

print("\nTotal Forecast Predictions:", len(sarima_tuning_predictions_df))
print("Total Fold-Horizon Metric Rows:", len(sarima_tuning_results_df))

assert len(sarima_tuning_predictions_df) == 1764
assert len(sarima_tuning_results_df) == 126

print("\nSARIMA Hyperparameter Tuning: Completed")

Configuration 1/6 completed | Order: (1, 0, 0) | Seasonal Order: (1, 0, 0, 6)
Configuration 2/6 completed | Order: (1, 0, 0) | Seasonal Order: (1, 0, 1, 6)
Configuration 3/6 completed | Order: (1, 0, 1) | Seasonal Order: (1, 0, 0, 6)
Configuration 4/6 completed | Order: (1, 0, 1) | Seasonal Order: (1, 0, 1, 6)
Configuration 5/6 completed | Order: (2, 0, 0) | Seasonal Order: (1, 0, 0, 6)
Configuration 6/6 completed | Order: (2, 0, 0) | Seasonal Order: (1, 0, 1, 6)

SARIMA Hyperparameter Tuning Results:


,Configuration,Order,Seasonal_Order,Mean_MAE,Mean_RMSE,Mean_R2
0,4,"(1, 0, 1)","(1, 0, 1, 6)",11895.1008,16305.0636,0.0284
1,3,"(1, 0, 1)","(1, 0, 0, 6)",12125.2043,16505.7735,-0.0108
2,1,"(1, 0, 0)","(1, 0, 0, 6)",12167.3510,16827.6427,-0.0100
3,5,"(2, 0, 0)","(1, 0, 0, 6)",12221.4138,16882.6956,-0.0249
4,2,"(1, 0, 0)","(1, 0, 1, 6)",12262.6614,16780.3836,-0.0190
5,6,"(2, 0, 0)","(1, 0, 1, 6)",12287.8989,16839.2867,-0.0295



Best SARIMA Parameters: {'order': (1, 0, 1), 'seasonal_order': (1, 0, 1, 6)}
Best Validation MAE: 11895.1008

Total Forecast Predictions: 1764
Total Fold-Horizon Metric Rows: 126

SARIMA Hyperparameter Tuning: Completed


**Key Observations**
- **Optimal SARIMA Configuration:** SARIMA achieved its lowest average MAE (11,895.10) using the nonseasonal order (1, 0, 1) and seasonal order (1, 0, 1, 6), outperforming the other five configurations.
- **Tuning Outcome:** The best configuration improved upon the original SARIMA benchmark (MAE: 11,980.85) by approximately 0.72%, while remaining slightly behind AutoReg (MAE: 11,812.55).

#### Tuning the Random Forest Regression Model

In [45]:
rf_tuning_results = []
rf_tuning_predictions = []

rf_configurations = list(
    ParameterGrid(hyperparameter_search_spaces["Random_Forest"])
)

for config_number, parameters in enumerate(
    rf_configurations,
    start=1
):
    for fold_number, (train_indices, validation_indices) in enumerate(
        validation_folds,
        start=1
    ):
        fold_predictions = []

        for horizon in range(1, forecast_horizon + 1):
            feature_columns = horizon_feature_columns[horizon]
            target_column = horizon_target_columns[horizon]

            X_train = train_df.iloc[train_indices][feature_columns]
            y_train = train_df.iloc[train_indices][target_column]

            X_validation = train_df.iloc[validation_indices][feature_columns]
            y_validation = train_df.iloc[validation_indices][target_column]

            model = RandomForestRegressor(
                **parameters,
                random_state=random_seed,
                n_jobs=-1
            )

            model.fit(X_train, y_train)

            predictions = model.predict(X_validation)

            assert np.isfinite(predictions).all()

            rf_tuning_results.append({
                "Model": "Random_Forest",
                "Configuration": config_number,
                "Fold": fold_number,
                "Horizon": horizon,
                "MAE": mean_absolute_error(
                    y_validation,
                    predictions
                ),
                "RMSE": np.sqrt(
                    mean_squared_error(
                        y_validation,
                        predictions
                    )
                ),
                "R2": r2_score(
                    y_validation,
                    predictions
                )
            })

            for sample_index, actual, predicted in zip(
                validation_indices,
                y_validation,
                predictions
            ):
                rf_tuning_predictions.append({
                    "Model": "Random_Forest",
                    "Configuration": config_number,
                    "Fold": fold_number,
                    "Horizon": horizon,
                    "Forecast_Origin": train_df.iloc[
                        sample_index
                    ]["Forecast_Origin"],
                    "Actual": float(actual),
                    "Predicted": float(predicted)
                })

    print(
        f"Configuration {config_number}/{len(rf_configurations)} completed"
    )

rf_tuning_results_df = pd.DataFrame(rf_tuning_results)
rf_tuning_predictions_df = pd.DataFrame(rf_tuning_predictions)

rf_tuning_summary_df = (
    rf_tuning_results_df
    .groupby("Configuration", as_index=False)
    .agg(
        Mean_MAE=("MAE", "mean"),
        Mean_RMSE=("RMSE", "mean"),
        Mean_R2=("R2", "mean")
    )
    .sort_values("Mean_MAE")
    .reset_index(drop=True)
)

rf_tuning_summary_df["Parameters"] = (
    rf_tuning_summary_df["Configuration"]
    .apply(lambda config: rf_configurations[int(config) - 1])
    .astype(str)
)

best_rf_configuration = int(
    rf_tuning_summary_df.iloc[0]["Configuration"]
)

best_rf_parameters = rf_configurations[
    best_rf_configuration - 1
]

print("\nRandom Forest Hyperparameter Tuning Results:")
display(rf_tuning_summary_df.round(4))

print("\nBest Random Forest Parameters:", best_rf_parameters)
print(
    "Best Validation MAE:",
    round(rf_tuning_summary_df.iloc[0]["Mean_MAE"], 4)
)

print("\nTotal Forecast Predictions:", len(rf_tuning_predictions_df))
print("Total Fold-Horizon Metric Rows:", len(rf_tuning_results_df))

assert len(rf_tuning_predictions_df) == 3528
assert len(rf_tuning_results_df) == 252

print("\nRandom Forest Hyperparameter Tuning: Completed")

Configuration 1/12 completed
Configuration 2/12 completed
Configuration 3/12 completed
Configuration 4/12 completed
Configuration 5/12 completed
Configuration 6/12 completed
Configuration 7/12 completed
Configuration 8/12 completed
Configuration 9/12 completed
Configuration 10/12 completed
Configuration 11/12 completed
Configuration 12/12 completed

Random Forest Hyperparameter Tuning Results:


,Configuration,Mean_MAE,Mean_RMSE,Mean_R2,Parameters
0,12,11987.4033,16809.6993,-0.0076,"{'max_depth': None, 'min_samples_leaf': 5, 'n_..."
1,8,11988.3040,16811.0576,-0.0078,"{'max_depth': 10, 'min_samples_leaf': 5, 'n_es..."
2,9,11993.0512,16698.3128,0.0007,"{'max_depth': None, 'min_samples_leaf': 2, 'n_..."
3,5,11996.3308,16695.5989,0.0016,"{'max_depth': 10, 'min_samples_leaf': 2, 'n_es..."
4,4,12008.8490,16847.4595,-0.0103,"{'max_depth': 5, 'min_samples_leaf': 5, 'n_est..."
5,11,12025.5869,16843.9058,-0.0114,"{'max_depth': None, 'min_samples_leaf': 5, 'n_..."
6,7,12026.9916,16845.1420,-0.0115,"{'max_depth': 10, 'min_samples_leaf': 5, 'n_es..."
7,10,12037.3397,16748.3620,-0.0058,"{'max_depth': None, 'min_samples_leaf': 2, 'n_..."
8,3,12043.5911,16871.4074,-0.0132,"{'max_depth': 5, 'min_samples_leaf': 5, 'n_est..."
9,6,12061.7673,16769.0874,-0.0083,"{'max_depth': 10, 'min_samples_leaf': 2, 'n_es..."



Best Random Forest Parameters: {'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 200}
Best Validation MAE: 11987.4033

Total Forecast Predictions: 3528
Total Fold-Horizon Metric Rows: 252

Random Forest Hyperparameter Tuning: Completed


**Key Observations**
- **Optimal Random Forest Configuration:** The best configuration used 200 estimators, unrestricted tree depth, and a minimum of five samples per leaf, achieving an average MAE of 11,987.40 and RMSE of 16,809.70.
- **Tuning Outcome:** Hyperparameter optimization reduced MAE by approximately 0.18% compared with the original Random Forest benchmark (12,008.94), indicating a modest improvement while AutoReg and tuned SARIMA remained more accurate.

#### Tuning the ElasticNet Regression Model

In [46]:
elasticnet_tuning_results = []
elasticnet_tuning_predictions = []

elasticnet_configurations = list(
    ParameterGrid(hyperparameter_search_spaces["ElasticNet"])
)

for config_number, parameters in enumerate(
    elasticnet_configurations,
    start=1
):
    for fold_number, (train_indices, validation_indices) in enumerate(
        validation_folds,
        start=1
    ):
        for horizon in range(1, forecast_horizon + 1):
            feature_columns = horizon_feature_columns[horizon]
            target_column = horizon_target_columns[horizon]

            X_train = train_df.iloc[train_indices][feature_columns]
            y_train = train_df.iloc[train_indices][target_column]

            X_validation = train_df.iloc[validation_indices][feature_columns]
            y_validation = train_df.iloc[validation_indices][target_column]

            model = Pipeline([
                ("scaler", StandardScaler()),
                ("regressor", ElasticNet(
                    **parameters,
                    max_iter=10000,
                    random_state=random_seed
                ))
            ])

            model.fit(X_train, y_train)

            predictions = model.predict(X_validation)

            assert np.isfinite(predictions).all()

            elasticnet_tuning_results.append({
                "Model": "ElasticNet",
                "Configuration": config_number,
                "Fold": fold_number,
                "Horizon": horizon,
                "MAE": mean_absolute_error(
                    y_validation,
                    predictions
                ),
                "RMSE": np.sqrt(
                    mean_squared_error(
                        y_validation,
                        predictions
                    )
                ),
                "R2": r2_score(
                    y_validation,
                    predictions
                )
            })

            for sample_index, actual, predicted in zip(
                validation_indices,
                y_validation,
                predictions
            ):
                elasticnet_tuning_predictions.append({
                    "Model": "ElasticNet",
                    "Configuration": config_number,
                    "Fold": fold_number,
                    "Horizon": horizon,
                    "Forecast_Origin": train_df.iloc[
                        sample_index
                    ]["Forecast_Origin"],
                    "Actual": float(actual),
                    "Predicted": float(predicted)
                })

    print(
        f"Configuration {config_number}/{len(elasticnet_configurations)} completed"
    )

elasticnet_tuning_results_df = pd.DataFrame(
    elasticnet_tuning_results
)

elasticnet_tuning_predictions_df = pd.DataFrame(
    elasticnet_tuning_predictions
)

elasticnet_tuning_summary_df = (
    elasticnet_tuning_results_df
    .groupby("Configuration", as_index=False)
    .agg(
        Mean_MAE=("MAE", "mean"),
        Mean_RMSE=("RMSE", "mean"),
        Mean_R2=("R2", "mean")
    )
    .sort_values("Mean_MAE")
    .reset_index(drop=True)
)

elasticnet_tuning_summary_df["Parameters"] = (
    elasticnet_tuning_summary_df["Configuration"]
    .apply(
        lambda config: elasticnet_configurations[int(config) - 1]
    )
    .astype(str)
)

best_elasticnet_configuration = int(
    elasticnet_tuning_summary_df.iloc[0]["Configuration"]
)

best_elasticnet_parameters = elasticnet_configurations[
    best_elasticnet_configuration - 1
]

print("\nElasticNet Hyperparameter Tuning Results:")
display(elasticnet_tuning_summary_df.round(4))

print("\nBest ElasticNet Parameters:", best_elasticnet_parameters)
print(
    "Best Validation MAE:",
    round(elasticnet_tuning_summary_df.iloc[0]["Mean_MAE"], 4)
)

print("\nTotal Forecast Predictions:", len(elasticnet_tuning_predictions_df))
print("Total Fold-Horizon Metric Rows:", len(elasticnet_tuning_results_df))

assert len(elasticnet_tuning_predictions_df) == 2646
assert len(elasticnet_tuning_results_df) == 189

print("\nElasticNet Hyperparameter Tuning: Completed")

Configuration 1/9 completed
Configuration 2/9 completed
Configuration 3/9 completed
Configuration 4/9 completed
Configuration 5/9 completed
Configuration 6/9 completed
Configuration 7/9 completed
Configuration 8/9 completed
Configuration 9/9 completed

ElasticNet Hyperparameter Tuning Results:


,Configuration,Mean_MAE,Mean_RMSE,Mean_R2,Parameters
0,4,12043.5372,16563.5542,0.0418,"{'alpha': 1.0, 'l1_ratio': 0.2}"
1,5,12078.3441,16616.4295,0.0322,"{'alpha': 1.0, 'l1_ratio': 0.5}"
2,9,12118.5255,16706.9057,0.0302,"{'alpha': 10.0, 'l1_ratio': 0.8}"
3,6,12278.2916,16869.1360,-0.0060,"{'alpha': 1.0, 'l1_ratio': 0.8}"
4,8,12328.8207,17159.3705,-0.0220,"{'alpha': 10.0, 'l1_ratio': 0.5}"
5,7,12476.7523,17459.5628,-0.0593,"{'alpha': 10.0, 'l1_ratio': 0.2}"
6,1,12583.9741,17151.4056,-0.0524,"{'alpha': 0.1, 'l1_ratio': 0.2}"
7,2,12725.6341,17260.2942,-0.0746,"{'alpha': 0.1, 'l1_ratio': 0.5}"
8,3,12931.6816,17390.3592,-0.1143,"{'alpha': 0.1, 'l1_ratio': 0.8}"



Best ElasticNet Parameters: {'alpha': 1.0, 'l1_ratio': 0.2}
Best Validation MAE: 12043.5372

Total Forecast Predictions: 2646
Total Fold-Horizon Metric Rows: 189

ElasticNet Hyperparameter Tuning: Completed


**Key Observations**
- **Optimal ElasticNet Configuration:** ElasticNet achieved its lowest average MAE (12,043.54) and RMSE (16,563.55) using an alpha of 1.0 and an L1 ratio of 0.2, with a positive mean R² of 0.0418.
- **Tuning Outcome:** Hyperparameter optimization reduced MAE by approximately 0.29% compared with the original ElasticNet benchmark (12,078.34), indicating a modest improvement through a greater emphasis on L2 regularization.

#### Tuning the Extra Trees Regression Model

In [47]:
extra_trees_tuning_results = []
extra_trees_tuning_predictions = []

extra_trees_configurations = list(
    ParameterGrid(hyperparameter_search_spaces["Extra_Trees"])
)

for config_number, parameters in enumerate(
    extra_trees_configurations,
    start=1
):
    for fold_number, (train_indices, validation_indices) in enumerate(
        validation_folds,
        start=1
    ):
        for horizon in range(1, forecast_horizon + 1):
            feature_columns = horizon_feature_columns[horizon]
            target_column = horizon_target_columns[horizon]

            X_train = train_df.iloc[train_indices][feature_columns]
            y_train = train_df.iloc[train_indices][target_column]

            X_validation = train_df.iloc[validation_indices][feature_columns]
            y_validation = train_df.iloc[validation_indices][target_column]

            model = ExtraTreesRegressor(
                **parameters,
                random_state=random_seed,
                n_jobs=-1
            )

            model.fit(X_train, y_train)

            predictions = model.predict(X_validation)

            assert np.isfinite(predictions).all()

            extra_trees_tuning_results.append({
                "Model": "Extra_Trees",
                "Configuration": config_number,
                "Fold": fold_number,
                "Horizon": horizon,
                "MAE": mean_absolute_error(
                    y_validation,
                    predictions
                ),
                "RMSE": np.sqrt(
                    mean_squared_error(
                        y_validation,
                        predictions
                    )
                ),
                "R2": r2_score(
                    y_validation,
                    predictions
                )
            })

            for sample_index, actual, predicted in zip(
                validation_indices,
                y_validation,
                predictions
            ):
                extra_trees_tuning_predictions.append({
                    "Model": "Extra_Trees",
                    "Configuration": config_number,
                    "Fold": fold_number,
                    "Horizon": horizon,
                    "Forecast_Origin": train_df.iloc[
                        sample_index
                    ]["Forecast_Origin"],
                    "Actual": float(actual),
                    "Predicted": float(predicted)
                })

    print(
        f"Configuration {config_number}/{len(extra_trees_configurations)} completed"
    )

extra_trees_tuning_results_df = pd.DataFrame(
    extra_trees_tuning_results
)

extra_trees_tuning_predictions_df = pd.DataFrame(
    extra_trees_tuning_predictions
)

extra_trees_tuning_summary_df = (
    extra_trees_tuning_results_df
    .groupby("Configuration", as_index=False)
    .agg(
        Mean_MAE=("MAE", "mean"),
        Mean_RMSE=("RMSE", "mean"),
        Mean_R2=("R2", "mean")
    )
    .sort_values("Mean_MAE")
    .reset_index(drop=True)
)

extra_trees_tuning_summary_df["Parameters"] = (
    extra_trees_tuning_summary_df["Configuration"]
    .apply(
        lambda config: extra_trees_configurations[int(config) - 1]
    )
    .astype(str)
)

best_extra_trees_configuration = int(
    extra_trees_tuning_summary_df.iloc[0]["Configuration"]
)

best_extra_trees_parameters = extra_trees_configurations[
    best_extra_trees_configuration - 1
]

print("\nExtra Trees Hyperparameter Tuning Results:")
display(extra_trees_tuning_summary_df.round(4))

print("\nBest Extra Trees Parameters:", best_extra_trees_parameters)
print(
    "Best Validation MAE:",
    round(extra_trees_tuning_summary_df.iloc[0]["Mean_MAE"], 4)
)

print("\nTotal Forecast Predictions:", len(extra_trees_tuning_predictions_df))
print("Total Fold-Horizon Metric Rows:", len(extra_trees_tuning_results_df))

assert len(extra_trees_tuning_predictions_df) == 3528
assert len(extra_trees_tuning_results_df) == 252

print("\nExtra Trees Hyperparameter Tuning: Completed")

Configuration 1/12 completed
Configuration 2/12 completed
Configuration 3/12 completed
Configuration 4/12 completed
Configuration 5/12 completed
Configuration 6/12 completed
Configuration 7/12 completed
Configuration 8/12 completed
Configuration 9/12 completed
Configuration 10/12 completed
Configuration 11/12 completed
Configuration 12/12 completed

Extra Trees Hyperparameter Tuning Results:


,Configuration,Mean_MAE,Mean_RMSE,Mean_R2,Parameters
0,3,11966.1197,16821.6208,-0.0186,"{'max_depth': 5, 'min_samples_leaf': 5, 'n_est..."
1,4,11984.7879,16837.9922,-0.0223,"{'max_depth': 5, 'min_samples_leaf': 5, 'n_est..."
2,12,12106.7522,16910.5137,-0.0422,"{'max_depth': None, 'min_samples_leaf': 5, 'n_..."
3,8,12112.2669,16930.8008,-0.0449,"{'max_depth': 10, 'min_samples_leaf': 5, 'n_es..."
4,11,12127.7426,16905.3970,-0.0433,"{'max_depth': None, 'min_samples_leaf': 5, 'n_..."
5,7,12134.1075,16937.0043,-0.0475,"{'max_depth': 10, 'min_samples_leaf': 5, 'n_es..."
6,2,12150.1182,16992.3143,-0.0587,"{'max_depth': 5, 'min_samples_leaf': 2, 'n_est..."
7,1,12169.7291,17020.7622,-0.0654,"{'max_depth': 5, 'min_samples_leaf': 2, 'n_est..."
8,10,12235.7445,17066.5813,-0.0906,"{'max_depth': None, 'min_samples_leaf': 2, 'n_..."
9,6,12256.2564,17054.6708,-0.0877,"{'max_depth': 10, 'min_samples_leaf': 2, 'n_es..."



Best Extra Trees Parameters: {'max_depth': 5, 'min_samples_leaf': 5, 'n_estimators': 100}
Best Validation MAE: 11966.1197

Total Forecast Predictions: 3528
Total Fold-Horizon Metric Rows: 252

Extra Trees Hyperparameter Tuning: Completed


**Key Observations**
- **Optimal Extra Trees Configuration:** Extra Trees achieved its lowest average MAE (11,966.12) and RMSE (16,821.62) using 100 estimators, a maximum tree depth of 5, and a minimum of five samples per leaf.
- **Tuning Outcome:** Hyperparameter optimization reduced MAE by approximately 2.47% compared with the original Extra Trees benchmark (12,269.25), indicating that shallower trees with stronger leaf-size constraints improved forecasting accuracy.

#### Comparing Hyperparameter Optimization Results

In [48]:
tuning_summary_sources = {
    "AutoReg": (
        autoreg_tuning_summary_df,
        best_autoreg_parameters
    ),
    "SARIMA": (
        sarima_tuning_summary_df,
        best_sarima_parameters
    ),
    "Random_Forest": (
        rf_tuning_summary_df,
        best_rf_parameters
    ),
    "ElasticNet": (
        elasticnet_tuning_summary_df,
        best_elasticnet_parameters
    ),
    "Extra_Trees": (
        extra_trees_tuning_summary_df,
        best_extra_trees_parameters
    )
}

tuning_comparison_records = []

for model_name, (summary_df, best_parameters) in tuning_summary_sources.items():
    original_result = all_model_comparison_df.loc[
        all_model_comparison_df["Model"] == model_name
    ].iloc[0]

    best_result = summary_df.iloc[0]

    original_mae = float(original_result["Mean_MAE"])
    tuned_mae = float(best_result["Mean_MAE"])

    tuning_comparison_records.append({
        "Model": model_name,
        "Original_MAE": original_mae,
        "Tuned_MAE": tuned_mae,
        "Tuned_RMSE": float(best_result["Mean_RMSE"]),
        "Tuned_R2": float(best_result["Mean_R2"]),
        "MAE_Improvement_%": (
            (original_mae - tuned_mae) / original_mae * 100
        ),
        "Best_Parameters": str(best_parameters)
    })

tuning_comparison_df = pd.DataFrame(
    tuning_comparison_records
)

tuning_comparison_df = (
    tuning_comparison_df
    .sort_values("Tuned_MAE")
    .reset_index(drop=True)
)

tuning_comparison_df.insert(
    0,
    "Rank",
    np.arange(1, len(tuning_comparison_df) + 1)
)

print("Hyperparameter Tuning Comparison:")
display(tuning_comparison_df.round(4))

print("\nBest Tuned Model:", tuning_comparison_df.iloc[0]["Model"])

print(
    "Best Tuned MAE:",
    round(tuning_comparison_df.iloc[0]["Tuned_MAE"], 4)
)

print(
    "\nModels Improved Through Tuning:",
    (tuning_comparison_df["MAE_Improvement_%"] > 0).sum()
)

assert len(tuning_comparison_df) == 5
assert tuning_comparison_df["Model"].is_unique

print("\nHyperparameter Tuning Comparison: Completed")

Hyperparameter Tuning Comparison:


,Rank,Model,Original_MAE,Tuned_MAE,Tuned_RMSE,Tuned_R2,MAE_Improvement_%,Best_Parameters
0,1,AutoReg,11812.5487,11812.5487,16419.5086,0.0166,0.0000,"{'lags': [1, 2, 3, 6, 12]}"
1,2,SARIMA,11980.8481,11895.1008,16305.0636,0.0284,0.7157,"{'order': (1, 0, 1), 'seasonal_order': (1, 0, ..."
2,3,Extra_Trees,12269.2499,11966.1197,16821.6208,-0.0186,2.4707,"{'max_depth': 5, 'min_samples_leaf': 5, 'n_est..."
3,4,Random_Forest,12008.9388,11987.4033,16809.6993,-0.0076,0.1793,"{'max_depth': None, 'min_samples_leaf': 5, 'n_..."
4,5,ElasticNet,12078.3441,12043.5372,16563.5542,0.0418,0.2882,"{'alpha': 1.0, 'l1_ratio': 0.2}"



Best Tuned Model: AutoReg
Best Tuned MAE: 11812.5487

Models Improved Through Tuning: 4

Hyperparameter Tuning Comparison: Completed


**Key Observations**
- **Tuned Model Performance:** AutoReg retained the lowest average MAE (11,812.55), followed by SARIMA (11,895.10) and Extra Trees (11,966.12), with all five tuned models outperforming the Seasonal Naive baseline.
- **Optimization Effectiveness:** Four of the five models achieved lower MAE after tuning, with Extra Trees recording the largest improvement (2.47%). AutoReg retained its original configuration, confirming its effectiveness within the evaluated search space.

#### Preparing Tuned Forecasts for Ensemble Development

In [49]:
tuned_prediction_sources = {
    "AutoReg": (
        autoreg_tuning_predictions_df,
        best_autoreg_configuration
    ),
    "SARIMA": (
        sarima_tuning_predictions_df,
        best_sarima_configuration
    ),
    "Random_Forest": (
        rf_tuning_predictions_df,
        best_rf_configuration
    ),
    "ElasticNet": (
        elasticnet_tuning_predictions_df,
        best_elasticnet_configuration
    ),
    "Extra_Trees": (
        extra_trees_tuning_predictions_df,
        best_extra_trees_configuration
    )
}

ensemble_keys = [
    "Fold",
    "Forecast_Origin",
    "Horizon"
]

ensemble_validation_df = None

for model_name, (predictions_df, configuration) in (
    tuned_prediction_sources.items()
):
    selected_predictions = (
        predictions_df.loc[
            predictions_df["Configuration"] == configuration,
            ensemble_keys + ["Actual", "Predicted"]
        ]
        .copy()
    )

    assert not selected_predictions.duplicated(
        ensemble_keys
    ).any()

    selected_predictions = selected_predictions.rename(
        columns={
            "Actual": f"Actual_{model_name}",
            "Predicted": model_name
        }
    )

    if ensemble_validation_df is None:
        ensemble_validation_df = selected_predictions
    else:
        ensemble_validation_df = ensemble_validation_df.merge(
            selected_predictions,
            on=ensemble_keys,
            how="inner",
            validate="one_to_one"
        )

actual_columns = [
    f"Actual_{model_name}"
    for model_name in tuned_prediction_sources
]

actual_values = ensemble_validation_df[
    actual_columns
].to_numpy(dtype=float)

assert np.allclose(
    actual_values,
    actual_values[:, [0]],
    rtol=1e-6,
    atol=1e-4
)

ensemble_validation_df["Actual"] = (
    ensemble_validation_df[actual_columns[0]]
)

ensemble_validation_df = (
    ensemble_validation_df
    .drop(columns=actual_columns)
    .sort_values(ensemble_keys)
    .reset_index(drop=True)
)

expected_columns = (
    ensemble_keys
    + ["Actual"]
    + list(tuned_prediction_sources)
)

ensemble_validation_df = ensemble_validation_df[
    expected_columns
]

print("Aligned Ensemble Validation Predictions:")
display(ensemble_validation_df.head(10).round(4))

print("\nPrediction Matrix Shape:", ensemble_validation_df.shape)
print("Models Available:", list(tuned_prediction_sources))
print("Validation Folds:", ensemble_validation_df["Fold"].nunique())
print("Forecast Horizons:", ensemble_validation_df["Horizon"].nunique())

assert len(ensemble_validation_df) == 294
assert ensemble_validation_df.shape[1] == 9
assert not ensemble_validation_df.isna().any().any()

print("\nEnsemble Prediction Alignment: Completed")

Aligned Ensemble Validation Predictions:


C:\Users\user\AppData\Local\Temp\ipykernel_5108\3933723300.py:102: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(ensemble_validation_df.head(10).round(4))


,Fold,Forecast_Origin,Horizon,Actual,AutoReg,SARIMA,Random_Forest,ElasticNet,Extra_Trees
0,1,2011-08-03,1,60241.75,31883.1357,32932.8755,28275.9903,27439.2302,29183.6609
1,1,2011-08-03,2,20824.67,22972.0114,23526.3516,24456.6091,25798.8116,26084.0697
2,1,2011-08-03,3,7464.12,26146.3177,26759.4408,13898.3906,16916.9032,15266.1260
3,1,2011-08-03,4,22667.22,24869.2874,24408.2596,24174.7429,25158.5597,24702.4366
4,1,2011-08-03,5,26661.47,24044.8315,25314.1848,28989.4512,24927.7103,30727.5665
5,1,2011-08-03,6,27871.61,25274.3466,25936.6240,28249.5338,24761.6170,28831.7473
6,1,2011-08-03,7,64737.77,31863.3943,29806.3506,26696.1258,25700.4004,26856.8822
7,1,2011-08-04,1,20824.67,25309.7188,27116.5864,32898.1222,27521.1038,31403.1316
8,1,2011-08-04,2,7464.12,24506.7070,24950.8827,15978.4642,19951.1006,14474.2802
9,1,2011-08-04,3,22667.22,26212.2564,25834.7177,31035.2801,28520.2430,31378.0284



Prediction Matrix Shape: (294, 9)
Models Available: ['AutoReg', 'SARIMA', 'Random_Forest', 'ElasticNet', 'Extra_Trees']
Validation Folds: 3
Forecast Horizons: 7

Ensemble Prediction Alignment: Completed


**Key Observations**
- **Forecast Alignment:** Predictions from all five tuned models were successfully aligned across 294 validation records, covering three expanding-window folds and seven forecast horizons without missing values.
- **Forecast Variability:** Individual models produce different revenue estimates for identical forecast targets, providing an opportunity to investigate whether combining their predictions improves forecasting accuracy.

#### Evaluating Voting Regression Ensembles

In [50]:
voting_ensemble_definitions = {
    "Voting_All_Models": [
        "AutoReg",
        "SARIMA",
        "Random_Forest",
        "ElasticNet",
        "Extra_Trees"
    ],
    "Voting_Statistical": [
        "AutoReg",
        "SARIMA"
    ],
    "Voting_Hybrid": [
        "AutoReg",
        "SARIMA",
        "Extra_Trees"
    ]
}

voting_validation_df = ensemble_validation_df[
    ["Fold", "Forecast_Origin", "Horizon", "Actual"]
].copy()

for ensemble_name, model_names in voting_ensemble_definitions.items():
    voting_validation_df[ensemble_name] = (
        ensemble_validation_df[model_names].mean(axis=1)
    )

voting_validation_metrics = []

for ensemble_name in voting_ensemble_definitions:
    for (fold, horizon), group in voting_validation_df.groupby(
        ["Fold", "Horizon"]
    ):
        voting_validation_metrics.append({
            "Model": ensemble_name,
            "Fold": fold,
            "Horizon": horizon,
            "MAE": mean_absolute_error(
                group["Actual"],
                group[ensemble_name]
            ),
            "RMSE": np.sqrt(
                mean_squared_error(
                    group["Actual"],
                    group[ensemble_name]
                )
            ),
            "R2": r2_score(
                group["Actual"],
                group[ensemble_name]
            )
        })

voting_validation_metrics_df = pd.DataFrame(
    voting_validation_metrics
)

voting_ensemble_summary_df = (
    voting_validation_metrics_df
    .groupby("Model", as_index=False)
    .agg(
        Mean_MAE=("MAE", "mean"),
        Mean_RMSE=("RMSE", "mean"),
        Mean_R2=("R2", "mean")
    )
    .sort_values("Mean_MAE")
    .reset_index(drop=True)
)

voting_ensemble_summary_df["Improvement_vs_AutoReg_%"] = (
    (
        tuning_comparison_df.iloc[0]["Tuned_MAE"]
        - voting_ensemble_summary_df["Mean_MAE"]
    )
    / tuning_comparison_df.iloc[0]["Tuned_MAE"]
    * 100
)

print("Voting Ensemble Validation Results:")
display(voting_ensemble_summary_df.round(4))

print("\nVoting Ensemble Forecast Samples:")
display(voting_validation_df.head(10).round(4))

print("\nBest Voting Ensemble:", voting_ensemble_summary_df.iloc[0]["Model"])
print(
    "Best Voting MAE:",
    round(voting_ensemble_summary_df.iloc[0]["Mean_MAE"], 4)
)

print("\nTotal Ensemble Forecast Records:", len(voting_validation_df))
print(
    "Total Fold-Horizon Metric Rows:",
    len(voting_validation_metrics_df)
)

assert len(voting_validation_df) == 294
assert len(voting_validation_metrics_df) == 63
assert voting_validation_df[
    list(voting_ensemble_definitions)
].notna().all().all()

print("\nVoting Ensemble Evaluation: Completed")

Voting Ensemble Validation Results:


,Model,Mean_MAE,Mean_RMSE,Mean_R2,Improvement_vs_AutoReg_%
0,Voting_Hybrid,11690.9452,16227.7544,0.0546,1.0294
1,Voting_All_Models,11697.9014,16278.9793,0.0605,0.9706
2,Voting_Statistical,11841.8191,16331.4210,0.0280,-0.2478



Voting Ensemble Forecast Samples:


C:\Users\user\AppData\Local\Temp\ipykernel_5108\1878189870.py:84: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(voting_validation_df.head(10).round(4))


,Fold,Forecast_Origin,Horizon,Actual,Voting_All_Models,Voting_Statistical,Voting_Hybrid
0,1,2011-08-03,1,60241.75,29942.9785,32408.0056,31333.2241
1,1,2011-08-03,2,20824.67,24567.5707,23249.1815,24194.1442
2,1,2011-08-03,3,7464.12,19797.4357,26452.8792,22723.9615
3,1,2011-08-03,4,22667.22,24662.6572,24638.7735,24659.9945
4,1,2011-08-03,5,26661.47,26800.7489,24679.5082,26695.5276
5,1,2011-08-03,6,27871.61,26610.7738,25605.4853,26680.9060
6,1,2011-08-03,7,64737.77,28184.6307,30834.8725,29508.8757
7,1,2011-08-04,1,20824.67,28849.7325,26213.1526,27943.1456
8,1,2011-08-04,2,7464.12,19972.2869,24728.7949,21310.6233
9,1,2011-08-04,3,22667.22,28596.1051,26023.4870,27808.3341



Best Voting Ensemble: Voting_Hybrid
Best Voting MAE: 11690.9452

Total Ensemble Forecast Records: 294
Total Fold-Horizon Metric Rows: 63

Voting Ensemble Evaluation: Completed


**Key Observations**
- **Best Voting Ensemble:** The Hybrid Voting ensemble achieved the lowest average MAE (11,690.95) and RMSE (16,227.75), outperforming the strongest individual model, AutoReg, by approximately 1.03% in MAE.
- **Ensemble Performance:** Both Hybrid Voting and All-Model Voting improved upon AutoReg, while Statistical Voting performed slightly worse, indicating that combining statistical and machine learning forecasts provided greater benefits than averaging statistical forecasts alone.

#### Preparing Chronological Stacking Validation Folds

In [51]:
stacking_feature_columns = [
    "AutoReg",
    "SARIMA",
    "Random_Forest",
    "ElasticNet",
    "Extra_Trees"
]

stacking_fold_partitions = []

for validation_fold in [2, 3]:
    validation_df = (
        ensemble_validation_df.loc[
            ensemble_validation_df["Fold"] == validation_fold
        ]
        .copy()
    )

    validation_start = validation_df["Forecast_Origin"].min()

    candidate_training_df = (
        ensemble_validation_df.loc[
            ensemble_validation_df["Fold"] < validation_fold
        ]
        .copy()
    )

    validation_start_position = trading_day_positions[
        pd.Timestamp(validation_start)
    ]

    candidate_training_df["Origin_Position"] = (
        candidate_training_df["Forecast_Origin"]
        .map(trading_day_positions)
    )

    training_df = candidate_training_df.loc[
        candidate_training_df["Origin_Position"] + forecast_horizon
        < validation_start_position
    ].drop(columns="Origin_Position")

    assert not training_df.empty
    assert not validation_df.empty

    assert (
        training_df["Forecast_Origin"].max()
        < validation_df["Forecast_Origin"].min()
    )

    assert (
        trading_day_positions[
            pd.Timestamp(training_df["Forecast_Origin"].max())
        ] + forecast_horizon
        < validation_start_position
    )

    stacking_fold_partitions.append({
        "Validation_Fold": validation_fold,
        "Training_Data": training_df,
        "Validation_Data": validation_df
    })

stacking_partition_summary_df = pd.DataFrame([
    {
        "Validation_Fold": partition["Validation_Fold"],
        "Training_Records": len(partition["Training_Data"]),
        "Validation_Records": len(partition["Validation_Data"]),
        "Training_Origins": partition["Training_Data"][
            "Forecast_Origin"
        ].nunique(),
        "Validation_Origins": partition["Validation_Data"][
            "Forecast_Origin"
        ].nunique(),
        "Training_End": partition["Training_Data"][
            "Forecast_Origin"
        ].max(),
        "Validation_Start": partition["Validation_Data"][
            "Forecast_Origin"
        ].min()
    }
    for partition in stacking_fold_partitions
])

print("Chronological Stacking Partitions:")
display(stacking_partition_summary_df)

print("\nStacking Features:", stacking_feature_columns)
print("Number of Stacking Validation Folds:", len(stacking_fold_partitions))

assert len(stacking_fold_partitions) == 2
assert stacking_partition_summary_df["Training_Records"].gt(0).all()

print("\nChronological Stacking Preparation: Completed")

Chronological Stacking Partitions:


,Validation_Fold,Training_Records,Validation_Records,Training_Origins,Validation_Origins,Training_End,Validation_Start
0,2,49,98,7,14,2011-08-10,2011-08-19
1,3,147,98,21,14,2011-08-26,2011-09-06



Stacking Features: ['AutoReg', 'SARIMA', 'Random_Forest', 'ElasticNet', 'Extra_Trees']
Number of Stacking Validation Folds: 2

Chronological Stacking Preparation: Completed


**Key Observations**
- **Chronological Validation:** Two stacking validation partitions were successfully prepared, with 49 training records for Fold 2 and 147 for Fold 3, each evaluated against 98 validation forecasts.
- **Leakage Prevention:** Training forecast origins were restricted to earlier periods, with overlapping seven-day target windows excluded before validation, preserving chronological separation between meta-model training and evaluation.

#### Training and Evaluating the Chronological Stacking Ensemble

In [52]:
stacking_validation_predictions = []
stacking_validation_metrics = []
stacking_fitted_models = {}

for partition in stacking_fold_partitions:
    validation_fold = partition["Validation_Fold"]
    meta_train_df = partition["Training_Data"]
    meta_validation_df = partition["Validation_Data"]

    X_meta_train = meta_train_df[stacking_feature_columns]
    y_meta_train = meta_train_df["Actual"]

    X_meta_validation = meta_validation_df[stacking_feature_columns]
    y_meta_validation = meta_validation_df["Actual"]

    stacking_model = Pipeline([
        ("scaler", StandardScaler()),
        ("regressor", Ridge(alpha=10.0))
    ])

    stacking_model.fit(X_meta_train, y_meta_train)

    stacking_predictions = stacking_model.predict(
        X_meta_validation
    )

    assert np.isfinite(stacking_predictions).all()

    stacking_fitted_models[validation_fold] = stacking_model

    fold_predictions_df = meta_validation_df[
        ["Fold", "Forecast_Origin", "Horizon", "Actual"]
    ].copy()

    fold_predictions_df["Stacking_Ridge"] = stacking_predictions

    stacking_validation_predictions.append(
        fold_predictions_df
    )

    for horizon, group in fold_predictions_df.groupby("Horizon"):
        stacking_validation_metrics.append({
            "Model": "Stacking_Ridge",
            "Fold": validation_fold,
            "Horizon": horizon,
            "MAE": mean_absolute_error(
                group["Actual"],
                group["Stacking_Ridge"]
            ),
            "RMSE": np.sqrt(
                mean_squared_error(
                    group["Actual"],
                    group["Stacking_Ridge"]
                )
            ),
            "R2": r2_score(
                group["Actual"],
                group["Stacking_Ridge"]
            )
        })

    print(
        f"Stacking Fold {validation_fold} completed | "
        f"Training Records: {len(meta_train_df)} | "
        f"Validation Records: {len(meta_validation_df)}"
    )

stacking_validation_predictions_df = pd.concat(
    stacking_validation_predictions,
    ignore_index=True
)

stacking_validation_metrics_df = pd.DataFrame(
    stacking_validation_metrics
)

stacking_summary_df = (
    stacking_validation_metrics_df
    .groupby("Model", as_index=False)
    .agg(
        Mean_MAE=("MAE", "mean"),
        Mean_RMSE=("RMSE", "mean"),
        Mean_R2=("R2", "mean")
    )
)

comparison_predictions_df = (
    stacking_validation_predictions_df
    .merge(
        voting_validation_df[
            [
                "Fold",
                "Forecast_Origin",
                "Horizon",
                "Voting_Hybrid"
            ]
        ],
        on=["Fold", "Forecast_Origin", "Horizon"],
        how="inner",
        validate="one_to_one"
    )
    .merge(
        ensemble_validation_df[
            [
                "Fold",
                "Forecast_Origin",
                "Horizon",
                "AutoReg"
            ]
        ],
        on=["Fold", "Forecast_Origin", "Horizon"],
        how="inner",
        validate="one_to_one"
    )
)

comparison_records = []

for model_name in [
    "Stacking_Ridge",
    "Voting_Hybrid",
    "AutoReg"
]:
    for (fold, horizon), group in comparison_predictions_df.groupby(
        ["Fold", "Horizon"]
    ):
        comparison_records.append({
            "Model": model_name,
            "Fold": fold,
            "Horizon": horizon,
            "MAE": mean_absolute_error(
                group["Actual"],
                group[model_name]
            ),
            "RMSE": np.sqrt(
                mean_squared_error(
                    group["Actual"],
                    group[model_name]
                )
            ),
            "R2": r2_score(
                group["Actual"],
                group[model_name]
            )
        })

stacking_fair_comparison_df = (
    pd.DataFrame(comparison_records)
    .groupby("Model", as_index=False)
    .agg(
        Mean_MAE=("MAE", "mean"),
        Mean_RMSE=("RMSE", "mean"),
        Mean_R2=("R2", "mean")
    )
    .sort_values("Mean_MAE")
    .reset_index(drop=True)
)

print("\nStacking Validation Results:")
display(stacking_summary_df.round(4))

print("\nStacking vs Voting vs AutoReg (Folds 2–3 Only):")
display(stacking_fair_comparison_df.round(4))

print("\nStacking Forecast Samples:")
display(comparison_predictions_df.head(10).round(4))

print(
    "\nBest Model on Comparable Folds:",
    stacking_fair_comparison_df.iloc[0]["Model"]
)

assert len(stacking_validation_predictions_df) == 196
assert len(stacking_validation_metrics_df) == 14
assert len(comparison_predictions_df) == 196
assert len(stacking_fair_comparison_df) == 3

print("\nChronological Stacking Evaluation: Completed")

Stacking Fold 2 completed | Training Records: 49 | Validation Records: 98
Stacking Fold 3 completed | Training Records: 147 | Validation Records: 98

Stacking Validation Results:


,Model,Mean_MAE,Mean_RMSE,Mean_R2
0,Stacking_Ridge,15337.9785,21245.9845,-1.376



Stacking vs Voting vs AutoReg (Folds 2–3 Only):


,Model,Mean_MAE,Mean_RMSE,Mean_R2
0,Voting_Hybrid,12419.7166,17597.9661,-0.0986
1,AutoReg,12750.7646,18139.1626,-0.1779
2,Stacking_Ridge,15337.9785,21245.9845,-1.3760



Stacking Forecast Samples:


C:\Users\user\AppData\Local\Temp\ipykernel_5108\1231836606.py:166: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(comparison_predictions_df.head(10).round(4))


,Fold,Forecast_Origin,Horizon,Actual,Stacking_Ridge,Voting_Hybrid,AutoReg
0,2,2011-08-19,1,14359.21,-10990.3065,15746.1673,12581.3316
1,2,2011-08-19,2,27776.41,19987.5310,25001.4962,22841.1387
2,2,2011-08-19,3,25424.54,23039.7296,25669.5137,24727.8591
3,2,2011-08-19,4,46683.96,50318.0165,31718.1274,32670.1248
4,2,2011-08-19,5,22369.22,72503.7437,38765.8122,41317.5607
5,2,2011-08-19,6,25260.13,13028.9213,22431.0171,22131.8962
6,2,2011-08-19,7,10784.78,-3385.3206,16731.1122,15725.0707
7,2,2011-08-21,1,27776.41,18204.1965,24121.5589,23028.9498
8,2,2011-08-21,2,25424.54,25306.9572,26666.0098,24520.4924
9,2,2011-08-21,3,46683.96,50647.2474,32322.4763,32634.9988



Best Model on Comparable Folds: Voting_Hybrid

Chronological Stacking Evaluation: Completed


**Key Observations**
- **Stacking Performance:** Ridge-based Stacking achieved an average MAE of 15,337.98 and RMSE of 21,245.98, substantially underperforming both Voting Hybrid and AutoReg across the comparable validation folds.
- **Ensemble Selection:** Voting Hybrid retained the lowest MAE (12,419.72) across Folds 2–3, while Stacking produced unstable forecasts, including negative revenue predictions, indicating limited generalization from the available meta-training data.